In [1]:
import pandas as pd
import numpy as np

In [2]:
import pyarrow.parquet as pq


In [3]:
base_KPIs = pd.read_csv('preprocessed_data/base_KPIs_labels_v3.csv')

In [4]:
base_KPIs


,account_no,year,respondent,maxcol,nchar_sum,prop_filled,environmental_claims_label_yes_sum,environmental_claims_label_yes_mean,environmental_claims_label_no_sum,environmental_claims_label_no_mean,...,transition_label_LABEL_0_sum,transition_label_LABEL_0_mean,transition_label_LABEL_1_sum,transition_label_LABEL_1_mean,transition_label_LABEL_2_sum,transition_label_LABEL_2_mean,renewable_label_LABEL_0_sum,renewable_label_LABEL_0_mean,renewable_label_LABEL_1_sum,renewable_label_LABEL_1_mean
0,200.0,2010,investor,191,4654,0.549738,2,0.010471,862,4.513089,...,22,0.115183,88,0.460733,2,0.010471,105,0.549738,5,0.026178
1,1800.0,2010,investor,191,13446,0.612565,4,0.020942,861,4.507853,...,31,0.162304,93,0.486911,4,0.020942,115,0.602094,16,0.083770
2,5300.0,2010,investor,191,62986,0.816754,1,0.005236,864,4.523560,...,50,0.261780,108,0.565445,11,0.057592,149,0.780105,20,0.104712
3,29900.0,2010,investor,191,51885,0.774869,7,0.036649,861,4.507853,...,50,0.261780,104,0.544503,9,0.047120,148,0.774869,7,0.036649
4,28600.0,2010,investor,191,6207,0.534031,2,0.010471,862,4.513089,...,28,0.146597,76,0.397906,2,0.010471,98,0.513089,9,0.047120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40999,34421.0,2020,supply_chain,434,20648,0.622120,10,0.023041,860,1.981567,...,66,0.152074,367,0.845622,4,0.009217,391,0.900922,42,0.096774
41000,838191.0,2020,supply_chain,434,4008,0.096774,0,0.000000,864,1.990783,...,11,0.025346,328,0.755760,1,0.002304,332,0.764977,7,0.016129
41001,37896.0,2020,supply_chain,434,13797,0.232719,14,0.032258,852,1.963134,...,32,0.073733,307,0.707373,1,0.002304,327,0.753456,11,0.025346
41002,71988.0,2020,supply_chain,434,4790,0.232719,1,0.002304,863,1.988479,...,18,0.041475,368,0.847926,1,0.002304,370,0.852535,16,0.036866


In [5]:
import time

start_time = time.time()
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")

df = pq.read_table(source="Chase Hikida/metrics.parquet").to_pandas() # takes about 6 minutes

end_time = time.time()
print(f"End: {time.strftime('%Y-%m-%d %H:%M:%S')}")

elapsed_seconds = end_time - start_time
elapsed_hms = time.strftime('%H:%M:%S', time.gmtime(elapsed_seconds))
print(f'Elapsed: {elapsed_hms}')
# 8:51

Start: 2026-03-02 14:38:50
End: 2026-03-02 14:45:02
Elapsed: 00:06:12


In [6]:
df.columns

def save_columns_to_csv(df, filename):
    pd.DataFrame({'Column': df.columns}).to_csv(filename, index=False)

# Usage:
save_columns_to_csv(df, 'Chase Hikida/metrics_columns.csv')

# Getting scope 1, 2, 3 emissions data --> save df

In [7]:
df["column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response"]

0        [The Greenhouse Gas Protocol: A Corporate Acco...
1                          [India GHG Inventory Programme]
2        [The Greenhouse Gas Protocol: A Corporate Acco...
3        [The Greenhouse Gas Protocol: A Corporate Acco...
4        [The Greenhouse Gas Protocol: A Corporate Acco...
                               ...                        
40999    [China Corporate Energy Conservation and GHG M...
41000               [ABI Energia Linee Guida; ISO 14064-1]
41001                                               [None]
41002    [China Corporate Energy Conservation and GHG M...
41003    [IPCC Guidelines for National Greenhouse Gas I...
Name: column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response, Length: 41004, dtype: object

In [9]:
# Explode lists into rows
df_exploded = df["column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response"].explode()

# Clean
df_clean = (df_exploded
    .str.strip()
    .str.lower()
    .replace('none', None)
    .dropna()
)

print(f"Unique responses: {df_clean.nunique()}")
print("\nTop 10:")
print(df_clean.value_counts().head(10))

Unique responses: 3064

Top 10:
column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response
the greenhouse gas protocol: a corporate accounting and reporting standard (revised edition)                                                                                                                      13796
other                                                                                                                                                                                                              5908
iso 14064-1                                                                                                                                                                                                        3223
defra voluntary reporting guidelines                                                                                                      

In [100]:
[col for col in df.columns if "Scope 3" in col]


['column=20.1C3. Scope 3 (Q15.1)|sheet=20.1A|value=response',
 'column=C6.5_C3_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Emissions calculation methodology|sheet=C6.5|value=response',
 'column=C4.1a_C4_Provide details of your absolute emissions target(s) and progress made against those targets. - Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response',
 'column=C4.1a_C7_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in base year as % of total base year emissions in selected Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response',
 'column=C4.1b_C4_Provide details of your emissions intensity target(s) and progress made against those target(s). - Scope(s) (or Scope 3 category)|sheet=C4.1b|value=response',
 'column=C4.1b_C8_Provide details of your emissions intensity target(s) and progress made against those target(s). - % of total base year emissions i

In [96]:
[col for col in df.columns if "column=C6.10_C2_Describe your gross global combined Scope 1 and 2 emissions for" in col]

# SCOPE 1 
# 'column=C6.1_C1_What were your organization’s gross global Scope 1 emissions in metric tons CO2e? - Gross global Scope 1 emissions (metric tons CO2e)|sheet=C6.1|value=response'
# 'column=8.2c. Please provide your gross global Scope 1 emissions figures in metric tonnes CO2e - Part 1 Total 8.2c. Gross global Scope 1 emissions (metric tonnes CO2e) – Part 1 Total|sheet=8.2c|value=response',
# 'column=8.2b. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2b|value=response'
# 'column=8.2d. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2d|value=response'
# 'column=CC9.2e. Scope 1 emissions (metric tonnes CO2e)|sheet=CC9.2e|value=response'

# SCOPE 2
# 'column=CC8.3. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e|sheet=CC8. Emissions Data|value=response'
#  'column=8.3c. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e - Part 1 Total 8.3c. Gross global Scope 2 emissions (metric tonnes CO2e) - Total Part 1|sheet=8.3c|value=response',
#  'column=C6.3_C1_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, location-based|sheet=C6.3|value=response',
#  'column=C6.3_C2_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, market-based (if applicable)|sheet=C6.3|value=response',

# SCOPE 1 + 2
#  'column=C6.10_C2_Describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tons CO2e per unit currency total revenue and provide any additional intensity metrics that are appropriate to your business operations. - Metric numerator (Gross global combined Scope 1 and 2 emissions, metric tons CO2e)|sheet=C6.10|value=response'
# 'column=CC12.2 C2 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Metric numerator (Gross global combined Scope 1 and 2 emissions)?|sheet=CC12.2|value=response'
# SCOPE 3
#  'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response',


# rename info cols if not done yet
# df.rename(columns={
#     'column=Account number|sheet=*|value=response': 'account_no',
#     'column=year|sheet=*|value=meta': 'year',
#     'column=label|sheet=*|value=meta': 'cdp_respondent', # already deciphered label as respondent
#     'column=Organization|sheet=Summary Data|value=response': 'cdp_name',
#     'column=Country|sheet=Summary Data|value=response': 'cdp_country',
#     'column=Primary industry|sheet=Summary Data|value=response': 'cdp_primary_industry'
# }, inplace=True) # the dataframe as it is, not make a copy to overflow memory!

df.rename(columns={
    'column=C6.1_C1_What were your organization’s gross global Scope 1 emissions in metric tons CO2e? - Gross global Scope 1 emissions (metric tons CO2e)|sheet=C6.1|value=response': 'cdp_scope1',
    'column=8.2c. Please provide your gross global Scope 1 emissions figures in metric tonnes CO2e - Part 1 Total 8.2c. Gross global Scope 1 emissions (metric tonnes CO2e) – Part 1 Total|sheet=8.2c|value=response': 'cdp_scope1_altcol',
    'column=8.2b. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2b|value=response': 'cdp_scope1_altcol2',
    'column=8.2d. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2d|value=response': 'cdp_scope1_altcol3',
    'column=CC9.2e. Scope 1 emissions (metric tonnes CO2e)|sheet=CC9.2e|value=response': 'cdp_scope1_altcol4',
    'column=CC8.3. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e|sheet=CC8. Emissions Data|value=response': 'cdp_scope2_unspec', # already deciphered label as respondent
    'column=8.3c. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e - Part 1 Total 8.3c. Gross global Scope 2 emissions (metric tonnes CO2e) - Total Part 1|sheet=8.3c|value=response': 'cdp_scope2_unspec_altcol',
    'column=C6.3_C1_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, location-based|sheet=C6.3|value=response': 'cdp_scope2_loc',
    'column=C6.3_C2_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, market-based (if applicable)|sheet=C6.3|value=response': 'cdp_scope2_mkt',
    'column=C6.10_C2_Describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tons CO2e per unit currency total revenue and provide any additional intensity metrics that are appropriate to your business operations. - Metric numerator (Gross global combined Scope 1 and 2 emissions, metric tons CO2e)|sheet=C6.10|value=response': 'cdp_scope12',
    'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response': 'cdp_scope3'
}, inplace=True) # the dataframe as it is, not make a copy to overflow memory!

info_cols = ['account_no', 'year', 'cdp_respondent', 'cdp_name', 'cdp_country', 'cdp_primary_industry']
emis_cols = ['cdp_scope1', 'cdp_scope1_altcol', 'cdp_scope1_altcol2',
             'cdp_scope1_altcol3', 'cdp_scope1_altcol4',
             'cdp_scope2_unspec', 'cdp_scope2_unspec_altcol', 
             'cdp_scope2_loc', 'cdp_scope2_mkt', 'cdp_scope12', 'cdp_scope3']                    

def function_to_clean_cells_and_set_index_types(df_input, numeric_col_list, string_col_list):
    
    total_cols = len(numeric_col_list)
    # for the core survey, just replace and index the account_no and year cols
    for i, colname in enumerate(numeric_col_list,1):

        df_input.loc[:,colname] = (
            df_input.loc[:,colname]
            .astype(str)
            .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
            .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
        )
        if i % 100 == 0 or i == total_cols:
            print(f"Processed {i} of {total_cols} columns")

    for colname in string_col_list:

        df_input.loc[:,colname] = (
            df_input.loc[:,colname]
            .astype(str)
            .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
        )

    # set data types and index
    df_input['account_no'] = df_input['account_no'].astype(str)
    df_input['year'] = df_input['year'].astype(int)
                    
    return df_input

# Subset the DataFrame
df_emissions = df[info_cols + emis_cols].copy()

                    
print(len(df_emissions.columns))
print(len(df_emissions))

df_emissions = function_to_clean_cells_and_set_index_types(df_emissions, list(set(['year'] + emis_cols)), 
                ['cdp_respondent', 'cdp_name', 'cdp_country', 'account_no', 'cdp_primary_industry'])


# faster indexing with account_no, year set as inplace index! use this to see which cols had highest scores
mask = (df_nz_scores['account_no'] == '943') # ARKEMA test
df_emissions.loc[:, info_cols+emis_cols].to_csv("Chase Hikida/cdp_emissions.csv", columns = info_cols+emis_cols, index = False, encoding="utf-8")
print('Saved cdp_emissions.csv')

17
41004
Processed 12 of 12 columns
Saved cdp_emissions.csv


In [97]:
display(df_emissions.loc[mask, info_cols+emis_cols].head(10))


,account_no,year,cdp_respondent,cdp_name,cdp_country,cdp_primary_industry,cdp_scope1,cdp_scope1_altcol,cdp_scope1_altcol2,cdp_scope1_altcol3,cdp_scope1_altcol4,cdp_scope2_unspec,cdp_scope2_unspec_altcol,cdp_scope2_loc,cdp_scope2_mkt,cdp_scope12,cdp_scope3
2538,943,2011,investor,ARKEMA,France,Chemicals,2770000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4823,943,2012,investor,ARKEMA,France,Chemicals,2090000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8072,943,2013,investor,ARKEMA,France,Chemicals,5120000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11576,943,2014,investor,ARKEMA,France,Chemicals,4710000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15552,943,2015,investor,ARKEMA,France,Chemicals,3430000.0,NaN,NaN,NaN,NaN,1067000.0,NaN,NaN,NaN,NaN,NaN
19555,943,2016,investor,ARKEMA,France,Chemicals,3000000.0,NaN,NaN,NaN,NaN,NaN,NaN,1300000.0,NaN,NaN,NaN
22563,943,2017,investor,ARKEMA,France,Chemicals,3110000.0,NaN,NaN,NaN,NaN,NaN,NaN,1080000.0,NaN,NaN,NaN
27477,943,2018,investor,ARKEMA,France,Manufacturing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4080000.0,NaN
29249,943,2018,supply_chain,ARKEMA,France,Manufacturing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4080000.0,NaN
30867,943,2019,investor,ARKEMA,France,Manufacturing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3875000.0,NaN


# Use base_KPIs_labels_v3.csv to filter for account_no's and years w each label 

In [30]:
# 2. Count of non-empty responses per Account number per column

# rename
df.rename(columns={
    'column=Account number|sheet=*|value=response': 'account_no',
    'column=year|sheet=*|value=meta': 'year',
    'column=label|sheet=*|value=meta': 'cdp_respondent', # already deciphered label as respondent
    'column=Organization|sheet=Summary Data|value=response': 'cdp_name',
    'column=Country|sheet=Summary Data|value=response': 'cdp_country',
    'column=Primary industry|sheet=Summary Data|value=response': 'cdp_primary_industry'
}, inplace=True) # the dataframe as it is, not make a copy to overflow memory!

info_cols = ['account_no', 'year', 'cdp_respondent', 'cdp_name', 'cdp_country', 'cdp_primary_industry']
                    
def function_to_clean_cells_and_set_index_types(df_input, numeric_col_list, string_col_list):
    
    total_cols = len(numeric_col_list)
    # for the core survey, just replace and index the account_no and year cols
    for i, colname in enumerate(numeric_col_list,1):

        df_input.loc[:,colname] = (
            df_input.loc[:,colname]
            .astype(str)
            .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
            .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
        )
        if i % 100 == 0 or i == total_cols:
            print(f"Processed {i} of {total_cols} columns")

    for colname in string_col_list:

        df_input.loc[:,colname] = (
            df_input.loc[:,colname]
            .astype(str)
            .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
        )

    # set data types and index
    df_input['account_no'] = df_input['account_no'].astype(str)
    df_input['year'] = df_input['year'].astype(int)
                    
    return df_input


                    
# now get scores!                      
nz_score = [col for col in df.columns if ("model=netzero_reduction|label=net-zero|value=score" in col)] # model= | label=net-zero
nz_score_subset = nz_score#[:2] # response_cols = [col for col in df.columns if ("value=response" in col)]

# Combine filtered columns with explicitly named columns and deduplicate
info_plus_nz_score = info_cols + nz_score_subset #+ response_cols 

# Subset the DataFrame
df_nz_scores = df[info_plus_nz_score].copy()

                    
print(len(df_nz_scores.columns))
print(len(df_nz_scores))

df_nz_scores = function_to_clean_cells_and_set_index_types(df_nz_scores, list(set(['year'] + nz_score_subset)), 
                ['cdp_respondent', 'cdp_name', 'cdp_country', 'account_no', 'cdp_primary_industry'])


# faster indexing with account_no, year set as inplace index! use this to see which cols had highest scores
mask = (df_nz_scores['account_no'] == '12117') & (df_nz_scores['year'] == 2020)
df_nz_scores.loc[mask, info_plus_nz_score].to_csv("Chase Hikida/12117_2020_NZscores.csv", columns = info_plus_nz_score, index = False, encoding="utf-8")
display(df_nz_scores.loc[mask, info_plus_nz_score].head())
print('Saved 12117_2020_NZscores.csv')

year
892
41004
column=CC12.2 C1 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Intensity figure =|sheet=CC12.2|model=netzero_reduction|label=net-zero|value=score
column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated responsibilities are, and how climate-related issues are monitored (do not include the names of individuals).|sheet=C1 - Governance|model=netzero_reduction|label=net-zero|value=score
column=C7.3b_C2_Break down your total gross global Scope 1 emissions by business facility. - Scope 1 emissions (metric tons CO2e)|sheet=C7.3b|model=netzero_reduction|label=net-zero|value=score
column=C9.1_C3_Provide any additional climate-related metrics relevant to your business. - Metric numerator|sheet=C9.1|model=netzero_reduction|label=net-zero|value=score
column=C12.3d_Do you publicly disclose a list of all research o

column=C10.1a_C4_Provide further details of the verification/assurance undertaken for your Scope 1 emissions, and attach the relevant statements. - Attach the statement|sheet=C10.1a|model=netzero_reduction|label=net-zero|value=score
column=C4.1a_C10_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in target year (metric tons CO2e) [auto-calculated]|sheet=C4.1a|model=netzero_reduction|label=net-zero|value=score
column=CC8.5. Scope 2 emissions: Main sources of uncertainty|sheet=CC8.5|model=netzero_reduction|label=net-zero|value=score
column=C10.1b_C2_Provide further details of the verification/assurance undertaken for your Scope 2 emissions and attach the relevant statements. - Verification or assurance cycle in place|sheet=C10.1b|model=netzero_reduction|label=net-zero|value=score
column=7.2C3. Timescale in Years|sheet=7.2A|model=netzero_reduction|label=net-zero|value=score
column=C4.2b_C1_Provide details of any other clima

column=C4.3b_C4_Provide details on the initiatives implemented in the reporting year in the table below. - Voluntary/Mandatory|sheet=C4.3b|model=netzero_reduction|label=net-zero|value=score
column=8.2d. Please provide your gross global Scope 1 emissions figures in metric tonnes CO2e - Part 2 8.2d. Gross global Scope 1 emissions (metric tonnes CO2e) - Other operationally controlled entities, activities or facilities|sheet=8.2d|model=netzero_reduction|label=net-zero|value=score
column=CC1.2a C3 - Please provide further details on the incentives provided for the management of climate change issues - Incentivized performance indicator|sheet=CC1.2a|model=netzero_reduction|label=net-zero|value=score
column=3.6. Describe any actions the company has taken or plans to take to manage or adapt to the risks that have been identified, including the cost of those actions.|sheet=Risks&amp;amp;Opps 3|model=netzero_reduction|label=net-zero|value=score
column=C4.1b_C10_Provide details of your emissions 

column=CC2.1b - Please describe how your risk and opportunity identification processes are applied at both company and asset level|sheet=CC2. Strategy|model=netzero_reduction|label=net-zero|value=score
column=C12.1a_C5_Provide details of your climate-related supplier engagement strategy. - % of supplier-related Scope 3 emissions as reported in C6.5|sheet=C12.1a|model=netzero_reduction|label=net-zero|value=score
column=CC8.5 C2 - Please estimate the level of uncertainty of the total gross global Scope 1 and 2 emissions figures that you have supplied and specify the sources of uncertainty in your data gathering, handling and calculations - Uncertainty range|sheet=CC8.5|model=netzero_reduction|label=net-zero|value=score
column=C4.2a_C8_Provide details of your target(s) to increase low-carbon energy consumption or production. - Metric (target numerator if reporting an intensity target)|sheet=C4.2a|model=netzero_reduction|label=net-zero|value=score
column=CC0.3 C1 - Country list configurati

column=C4.1b_C12_Provide details of your emissions intensity target(s) and progress made against those target(s). - % change anticipated in absolute Scope 1+2 emissions|sheet=C4.1b|model=netzero_reduction|label=net-zero|value=score
column=9.7 C11. 11. Monetary savings|sheet=9.7|model=netzero_reduction|label=net-zero|value=score
column=Do you want to answer using:|sheet=Risks&amp;amp;Opps 8|model=netzero_reduction|label=net-zero|value=score
column=CC11.5 C2 - Please report how much electricity you produce in MWh, and how much electricity you consume in MWh? - Consumed electricity that is purchased (MWh)|sheet=CC11.5|model=netzero_reduction|label=net-zero|value=score
column=C2.4_Have you identified any climate-related opportunities with the potential to have a substantive financial or strategic impact on your business?|sheet=C2 - Risks and Opportunities|model=netzero_reduction|label=net-zero|value=score
column=C12.1b_C4_Give details of your climate-related engagement strategy with your c

...

column=CC12.2 C7 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Reason for change|sheet=CC12.2|model=netzero_reduction|label=net-zero|value=score
column=CC5.1b C8 - Please describe your inherent risks that are driven by changes in physical climate parameters - Estimated financial implications|sheet=CC5.1b|model=netzero_reduction|label=net-zero|value=score
column=C2.3a_C11_Provide details of risks identified with the potential to have a substantive financial or strategic impact on your business. - Potential financial impact figure (currency)|sheet=C2.3a|model=netzero_reduction|label=net-zero|value=score
column=CC14.3a C1 - Please identify the reasons for any change in your Scope 3 emissions and for each of them specify how your emissions compare to the previous year - Sources of Scope 3 emissions|sheet=CC14.3a|model=netzero_reduction|label=net-zero|value=score
column=2.1a. Please provide

column=C6.10_C4_Describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tons CO2e per unit currency total revenue and provide any additional intensity metrics that are appropriate to your business operations. - Metric denominator: Unit total|sheet=C6.10|model=netzero_reduction|label=net-zero|value=score
column=C10.1b_C6_Provide further details of the verification/assurance undertaken for your Scope 2 emissions and attach the relevant statements. - Page/ section reference|sheet=C10.1b|model=netzero_reduction|label=net-zero|value=score
column=C12.3e_Provide details of the other engagement activities that you undertake.|sheet=C12 - Engagement|model=netzero_reduction|label=net-zero|value=score
column=C6.10_C5_Describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tons CO2e per unit currency total revenue and provide any additional intensity metrics that are appropriate to your business operations. - Scope 2 figure u

column=C8.2d_C1_Provide details on the electricity, heat, steam, and cooling your organization has generated and consumed in the reporting year. - Total Gross generation (MWh)|sheet=C8.2d|model=netzero_reduction|label=net-zero|value=score
column=5.6. Describe any actions the company has taken or plans to take to manage or adapt to the other risks that have been identified, including the costs of those actions.|sheet=Risks&amp;amp;Opps 5|model=netzero_reduction|label=net-zero|value=score
column=C7.2_C2_Break down your total gross global Scope 1 emissions by country/region. - Scope 1 emissions (metric tons CO2e)|sheet=C7.2|model=netzero_reduction|label=net-zero|value=score
Index(['account_no', 'year', 'cdp_respondent', 'cdp_name', 'cdp_country',
       'cdp_primary_industry',
       'column=year|sheet=*|model=netzero_reduction|label=net-zero|value=score',
       'column=label|sheet=*|model=netzero_reduction|label=net-zero|value=score',
       'column=Organization|sheet=Summary Data|model

,account_no,year,cdp_respondent,cdp_name,cdp_country,cdp_primary_industry,column=year|sheet=*|model=netzero_reduction|label=net-zero|value=score,column=label|sheet=*|model=netzero_reduction|label=net-zero|value=score,column=Organization|sheet=Summary Data|model=netzero_reduction|label=net-zero|value=score,column=Account number|sheet=*|model=netzero_reduction|label=net-zero|value=score,...,"column=C4.2b_C6_Provide details of any other climate-related targets, including methane reduction targets. - Target denominator (intensity targets only)|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score","column=C4.2b_C11_Provide details of any other climate-related targets, including methane reduction targets. - Figure or percentage in reporting year|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score","column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score",column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|model=netzero_reduction|label=net-zero|value=score,column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|model=netzero_reduction|label=net-zero|value=score,column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|model=netzero_reduction|label=net-zero|value=score,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|model=netzero_reduction|label=net-zero|value=score",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|model=netzero_reduction|label=net-zero|value=score,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|model=netzero_reduction|label=net-zero|value=score,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|model=netzero_reduction|label=net-zero|value=score
36760,12117,2020,investor,MITIE Group,United Kingdom of Great Britain and Northern I...,Services,0.004353,0.00054,0.00087,0.002623,...,0.000536,0.001512,0.971191,0.001451,NaN,NaN,0.016556,NaN,NaN,NaN
39546,12117,2020,supply_chain,MITIE Group,United Kingdom of Great Britain and Northern I...,Services,0.004353,0.000793,0.00087,0.002623,...,0.000536,0.001512,0.971191,0.001451,NaN,NaN,0.016556,NaN,NaN,NaN


Saved 12117_2020_NZscores.csv
Saved 12117_2020_NZscores_topexamples.csv


In [64]:
# Filter columns by required patterns, get the scores in there for the final output to have answer + score
nz_scores = [col for col in df.columns if ("model=netzero_reduction|label=net-zero|value=score" in col)] # scores
value_cols = [col for col in df.columns if ("value=response" in col)] # text responses

df_core_survey = df[info_cols + value_cols + nz_scores].copy()
df_core_survey = function_to_clean_cells_and_set_index_types(df_core_survey, ['year'], ['cdp_respondent', 'cdp_name', 
                    'cdp_country', 'account_no', 'cdp_primary_industry'])

print(len(df_core_survey.columns))
print(len(df_core_survey))

year
1772
41004


In [68]:
# intermediate step of extracting column names with > 0.3 scores
row = df_nz_scores.loc[mask, info_plus_nz_score]  # select only relevant columns
row_t = row.transpose()                           # Transpose so values are now in a column
display(row_t.head(10))
cdp_vals = row_t.loc['cdp_respondent'].values
print(cdp_vals)

# Slice to exclude the first len(info_cols) rows
row_t_filtered = row_t.iloc[len(info_cols):, :]

# Dynamically rename the column using f-string
row_t_filtered.columns = [f'nz_score_{val}' for val in cdp_vals]

sorted_row = row_t_filtered.sort_values(f'nz_score_{cdp_vals[0]}', ascending=False)  # Sort descending
display(sorted_row.head(20))

cols_above_threshold = sorted_row[sorted_row[f'nz_score_{cdp_vals[0]}'] > 0.3].index.tolist()  # Get col names

print(cols_above_threshold)

,36760,39546
account_no,12117,12117
year,2020,2020
cdp_respondent,investor,supply_chain
cdp_name,MITIE Group,MITIE Group
cdp_country,United Kingdom of Great Britain and Northern I...,United Kingdom of Great Britain and Northern I...
cdp_primary_industry,Services,Services
column=year|sheet=*|model=netzero_reduction|label=net-zero|value=score,0.004353,0.004353
column=label|sheet=*|model=netzero_reduction|label=net-zero|value=score,0.00054,0.000793
column=Organization|sheet=Summary Data|model=netzero_reduction|label=net-zero|value=score,0.00087,0.00087
column=Account number|sheet=*|model=netzero_reduction|label=net-zero|value=score,0.002623,0.002623


['investor' 'supply_chain']


,nz_score_investor,nz_score_supply_chain
column=C3.1f_Provide any additional information on how climate-related risks and opportunities have influenced your strategy and financial planning (optional).|sheet=C3 - Business Strategy|model=netzero_reduction|label=net-zero|value=score,0.998047,0.998047
column=C4.2a_C19_Provide details of your target(s) to increase low-carbon energy consumption or production. - Please explain (including target coverage)|sheet=C4.2a|model=netzero_reduction|label=net-zero|value=score,0.997559,0.997559
column=C3.1c_Why does your organization not use climate-related scenario analysis to inform its strategy?|sheet=C3 - Business Strategy|model=netzero_reduction|label=net-zero|value=score,0.997559,0.997559
"column=C4.2b_C16_Provide details of any other climate-related targets, including methane reduction targets. - Please explain (including target coverage)|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score",0.997559,0.997559
"column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated responsibilities are, and how climate-related issues are monitored (do not include the names of individuals).|sheet=C1 - Governance|model=netzero_reduction|label=net-zero|value=score",0.988281,0.988281
"column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score",0.971191,0.971191
column=C4.1a_C15_Provide details of your absolute emissions target(s) and progress made against those targets. - Please explain (including target coverage)|sheet=C4.1a|model=netzero_reduction|label=net-zero|value=score,0.898926,0.898926
column=C4.1b_C18_Provide details of your emissions intensity target(s) and progress made against those target(s). - Please explain (including target coverage)|sheet=C4.1b|model=netzero_reduction|label=net-zero|value=score,0.898926,0.898926
column=C4.3c_C2_What methods do you use to drive investment in emissions reduction activities? - Comment|sheet=C4.3c|model=netzero_reduction|label=net-zero|value=score,0.894043,0.894043
"column=C12.1b_C7_Give details of your climate-related engagement strategy with your customers. - Impact of engagement, including measures of success|sheet=C12.1b|model=netzero_reduction|label=net-zero|value=score",0.880371,0.880371


['column=C3.1f_Provide any additional information on how climate-related risks and opportunities have influenced your strategy and financial planning (optional).|sheet=C3 - Business Strategy|model=netzero_reduction|label=net-zero|value=score', 'column=C4.2a_C19_Provide details of your target(s) to increase low-carbon energy consumption or production. - Please explain (including target coverage)|sheet=C4.2a|model=netzero_reduction|label=net-zero|value=score', 'column=C3.1c_Why does your organization not use climate-related scenario analysis to inform its strategy?|sheet=C3 - Business Strategy|model=netzero_reduction|label=net-zero|value=score', 'column=C4.2b_C16_Provide details of any other climate-related targets, including methane reduction targets. - Please explain (including target coverage)|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score', 'column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated 

In [67]:
# function to extract column base names
# extract {column_base_name}|model=...|label=...|value=score
# save as {column_base_name}|value=response
def extract_column_base_names(names_list):
    return [item.split('|model=')[0] + "|value=response" for item in names_list]

# columns with highish scores
# concerns about special characters
# pasted transposed and sorted by descending values, then selected colnames with values > 0.3
base_names = extract_column_base_names(cols_above_threshold)
print(base_names)

# save specific column responses to csv
specific_column_list = base_names
core_mask = (df_core_survey['account_no'] == '12117') & (df_core_survey['year'] == 2020)
final_cols = info_cols + sorted(specific_column_list + cols_above_threshold)
df_core_survey.loc[core_mask, final_cols].to_csv("Chase Hikida/12117_2020_NZscores_topexamples.csv", columns = final_cols, index=False)
display(df_core_survey.loc[core_mask, final_cols].head())
print('Saved 12117_2020_NZscores_topexamples.csv')


['column=C3.1f_Provide any additional information on how climate-related risks and opportunities have influenced your strategy and financial planning (optional).|sheet=C3 - Business Strategy|value=response', 'column=C4.2a_C19_Provide details of your target(s) to increase low-carbon energy consumption or production. - Please explain (including target coverage)|sheet=C4.2a|value=response', 'column=C3.1c_Why does your organization not use climate-related scenario analysis to inform its strategy?|sheet=C3 - Business Strategy|value=response', 'column=C4.2b_C16_Provide details of any other climate-related targets, including methane reduction targets. - Please explain (including target coverage)|sheet=C4.2b|value=response', 'column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated responsibilities are, and how climate-related issues are monitored (do not include the names of individuals).|sheet=C1 - Governance|value=respon

,account_no,year,cdp_respondent,cdp_name,cdp_country,cdp_primary_industry,column=C0.1_Give a general description and introduction to your organization.|sheet=C0 - Introduction|model=netzero_reduction|label=net-zero|value=score,column=C0.1_Give a general description and introduction to your organization.|sheet=C0 - Introduction|value=response,"column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated responsibilities are, and how climate-related issues are monitored (do not include the names of individuals).|sheet=C1 - Governance|model=netzero_reduction|label=net-zero|value=score","column=C1.2a_Describe where in the organizational structure this/these position(s) and/or committees lie, what their associated responsibilities are, and how climate-related issues are monitored (do not include the names of individuals).|sheet=C1 - Governance|value=response",...,column=C4.2a_C19_Provide details of your target(s) to increase low-carbon energy consumption or production. - Please explain (including target coverage)|sheet=C4.2a|model=netzero_reduction|label=net-zero|value=score,column=C4.2a_C19_Provide details of your target(s) to increase low-carbon energy consumption or production. - Please explain (including target coverage)|sheet=C4.2a|value=response,"column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score","column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|value=response","column=C4.2b_C16_Provide details of any other climate-related targets, including methane reduction targets. - Please explain (including target coverage)|sheet=C4.2b|model=netzero_reduction|label=net-zero|value=score","column=C4.2b_C16_Provide details of any other climate-related targets, including methane reduction targets. - Please explain (including target coverage)|sheet=C4.2b|value=response",column=C4.3b_C9_Provide details on the initiatives implemented in the reporting year in the table below. - Comment|sheet=C4.3b|model=netzero_reduction|label=net-zero|value=score,column=C4.3b_C9_Provide details on the initiatives implemented in the reporting year in the table below. - Comment|sheet=C4.3b|value=response,column=C4.3c_C2_What methods do you use to drive investment in emissions reduction activities? - Comment|sheet=C4.3c|model=netzero_reduction|label=net-zero|value=score,column=C4.3c_C2_What methods do you use to drive investment in emissions reduction activities? - Comment|sheet=C4.3c|value=response
36760,12117,2020,investor,MITIE Group,United Kingdom of Great Britain and Northern I...,Services,[0.538086],"[Founded in 1987, Mitie is one of the UK's lea...",[0.988281],[Chief Executive Officer – Overall responsibil...,...,[0.997559],[We have signed up to RE100 as part of our net...,[0.971191],"[Yes, converting our fleet to EV directly impa...",[0.997559],[We have signed up to EV100 as part of our net...,[0.398926],[We are converting our >5000 diesel fleet to e...,[0.894043],[Due to the nature of the work Mitie undertake...
39546,12117,2020,supply_chain,MITIE Group,United Kingdom of Great Britain and Northern I...,Services,[0.538086],"[Founded in 1987, Mitie is one of the UK's lea...",[0.988281],[Chief Executive Officer – Overall responsibil...,...,[0.997559],[We have signed up to RE100 as part of our net...,[0.971191],"[Yes, converting our fleet to EV directly impa...",[0.997559],[We have signed up to EV100 as part of our net...,[0.398926],[We are converting our >5000 diesel fleet to e...,[0.894043],[Due to the nature of the work Mitie undertake...


Saved 12117_2020_NZscores_topexamples.csv


# summary tables of cdp things

In [87]:
# filtering option
# (df_core_survey['year'] == 2020) & 
core_mask = (df_core_survey['account_no'].astype(str).isin(
    ['1104', '4657', '5052', '12117', '12942', '16012', '16558', '19051', '36707','52633', '60580', '832087']))

final_cols = info_cols
df_core_survey.loc[core_mask, final_cols].to_csv("Chase Hikida/2020_unique_cdp_industries.csv", columns = final_cols, index=False)
display(df_core_survey.loc[core_mask, final_cols].head())
print('Saved 2020_unique_cdp_industries.csv')

industry_by_year = (
    df_core_survey#.loc[core_mask, final_cols]
    .groupby('year')['cdp_primary_industry']
    .unique()
)

# Assuming industry_by_year is a Series mapping years to lists/arrays of industries
industry_table = industry_by_year.reset_index()
industry_table.columns = ['year', 'unique_industries']
display(industry_table)

industry_long = industry_by_year.explode().reset_index()
display(industry_long)

industry_account_counts = (
    df_core_survey[df_core_survey['year']==2020]
    .groupby(['year', 'cdp_primary_industry'])['account_no']
    .nunique()
    .reset_index(name='unique_account_count')
)
display(industry_account_counts)

industry_count_by_year = (
    df_core_survey
    .groupby('year')['cdp_primary_industry']
    .nunique()
)

print(industry_count_by_year)

,account_no,year,cdp_respondent,cdp_name,cdp_country,cdp_primary_industry
58,1104,2010,investor,AstraZeneca,United Kingdom,"Pharmaceuticals, Biotechnology and Life Sciences"
178,16012,2010,investor,Royal Dutch Shell,Netherlands,Oil and Gas
262,12117,2010,investor,MITIE Group,United Kingdom,"Trading Companies and Distributors, and Commer..."
624,12942,2010,investor,Nestle,Switzerland,Food and Beverage Processing
772,19051,2010,investor,The Westpac Group,Australia,"Banks, Diverse Financials, and Insurance"


Saved 2020_unique_cdp_industries.csv


,year,unique_industries
0,2010,[Electric Utilities & Independent Power Produc...
1,2011,"[Oil and Gas, Electric Utilities & Independent..."
2,2012,"[Technology Hardware and Equipment, Electric U..."
3,2013,"[Trading Companies and Distributors, and Comme..."
4,2014,"[Automobiles and Components, Banks, Diverse Fi..."
5,2015,"[Hotels, Restaurants & Leisure, and Tourism Se..."
6,2016,"[Hotels, Restaurants & Leisure, and Tourism Se..."
7,2017,"[Banks, Diverse Financials, Insurance, Ground ..."
8,2018,"[Services, Manufacturing, Power generation, Bi..."
9,2019,"[Services, Manufacturing, Food, beverage & agr..."


,year,cdp_primary_industry
0,2010,Electric Utilities & Independent Power Produce...
1,2010,Oil and Gas
2,2010,"Banks, Diverse Financials, and Insurance"
3,2010,"Textiles, Apparel, Footwear and Luxury Goods"
4,2010,To be categorized
...,...,...
463,2020,Power generation
464,2020,Retail
465,2020,Apparel
466,2020,Fossil Fuels


,year,cdp_primary_industry,unique_account_count
0,2020,Apparel,114
1,2020,"Biotech, health care & pharma",135
2,2020,"Food, beverage & agriculture",284
3,2020,Fossil Fuels,96
4,2020,Hospitality,40
5,2020,Infrastructure,238
6,2020,International bodies,1
7,2020,Manufacturing,1666
8,2020,Materials,477
9,2020,Power generation,75


year
2010    49
2011    48
2012    48
2013    47
2014    79
2015    49
2016    52
2017    52
2018    16
2019    15
2020    13
Name: cdp_primary_industry, dtype: int64


# other

In [13]:
df.tail()

,column=year|sheet=*|value=meta,column=label|sheet=*|value=meta,column=Organization|sheet=Summary Data|value=response,column=Account number|sheet=*|value=response,column=Discloser ID|sheet=Summary|value=response,column=Country|sheet=Summary Data|value=response,column=Primary industry|sheet=Summary Data|value=response,column=Primary Expansion|sheet=Summary|value=response,column=secondary_expansion|sheet=Summary Data|value=response,column=Complexity|sheet=Summary|value=response,...,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=number_responses",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_characters,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_responses,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_characters,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_responses,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|model=climate_specificity|label=spec|value=score_average,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_characters,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_responses
40999,[2020],[supply_chain],[ZINWELL CORPORATION],[34421],None,[China],[Manufacturing],None,None,None,...,1,0.439978,764.0,34,0.440581,769.0,34,0.438721,782.0,34
41000,[2020],[supply_chain],[ZKL PRINTING],[838191],None,[China],[Manufacturing],None,None,None,...,1,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34
41001,[2020],[supply_chain],[Zones],[37896],None,[United States of America],[Services],None,None,None,...,1,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34
41002,[2020],[supply_chain],[ZOTEQ],[71988],None,[China],[Materials],None,None,None,...,0,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34
41003,[2020],[supply_chain],[Zurich Insurance Group],[21064],None,[Switzerland],[Services],None,None,None,...,1,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34


In [7]:
df.tail()[['column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=response']]

# [Question not applicable, Question not applica...


,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=response
40999,"[Question not applicable, Question not applica..."
41000,"[Question not applicable, Question not applica..."
41001,"[Question not applicable, Question not applica..."
41002,"[Question not applicable, Question not applica..."
41003,"[Question not applicable, Question not applica..."


In [8]:
# df.to_csv("preprocessed_data/metrics.csv", index=False, chunksize=100) # 3.5hrs (9:30pm-1am) 14GB!

### thank you perplexity, start and end time; planning out next steps
import time

start_time = time.strftime("%Y-%m-%d %H:%M:%S")
print(f"Start time: {start_time}")

end_time = time.strftime("%Y-%m-%d %H:%M:%S")
print(f"End time: {end_time}")

next, try lambda lambda timing & crashing
then try streaming dataframe handling. 
if streaming doesn't work, then try column-wise explosion, keeping account_no and year, and then group_by later.

In [11]:
df.rename(columns={
    'column=Account number|sheet=*|value=response': 'account_no',
    'column=year|sheet=*|value=meta': 'year',
    'column=label|sheet=*|value=meta': 'respondent',
    'column=Organization|sheet=Summary Data|value=response': 'name'
}, inplace=True) # the dataframe as it is, not make a copy to overflow memory!

In [12]:
# find scope 1, 2 columns

df_core_survey = df[[col for col in df.columns if "column=" in col
  and "sheet=Summary" not in col 
  and "model=" not in col
  and "number_characters" not in col
  and "number_responses" not in col
 ] + ['account_no', 'year', 'name', 'respondent']] # + maxcol




In [13]:
[col for col in df_core_survey.columns 
     if "protocol" in col.lower()]

# initiative_details: 'column=C4.3b_C2_Provide details on the initiatives implemented in the reporting year in the table below. - Description of initiative|sheet=C4.3b|value=response'
# initiative_comment: 'column=C4.3b_C9_Provide details on the initiatives implemented in the reporting year in the table below. - Comment|sheet=C4.3b|value=response'
# change_dir: 'column=CC12.2 C6 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Direction of change from previous year|sheet=CC12.2|value=response'
# change_dir: 'column=CC12.3. Direction of change from previous year|sheet=CC12.3|value=response'
# change_dir: 'column=C7.9a_C2_Identify the reasons for any change in your gross global emissions (Scope 1 and 2 combined), and for each of them specify how your emissions compare to the previous year. - Direction of change|sheet=C7.9a|value=response'
# initiative_type: 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type|sheet=C4.3b|value=response'
# initiative_type: 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response'
# why_no_initiatives: 'column=C4.3d_Why did you not have any emissions reduction initiatives active during the reporting year?|sheet=C4 - Targets and Performance|value=response'
# absolute_target_scope: 'column=C4.1a_C4_Provide details of your absolute emissions target(s) and progress made against those targets. - Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response'
# intensity_target_scope: 'column=C4.1b_C4_Provide details of your emissions intensity target(s) and progress made against those target(s). - Scope(s) (or Scope 3 category)|sheet=C4.1b|value=response'
# absolute_target_comment: 'column=CC3.1a C9 - Please provide details of your absolute target - Comment|sheet=CC3.1a|value=response'
# intensity_target_comment: 'column=CC3.1b C10 - Please provide details of your intensity target - Comment|sheet=CC3.1b|value=response'
# science_based_absolute_target: 'column=C4.1a_C14_Provide details of your absolute emissions target(s) and progress made against those targets. - Is this a science-based target?|sheet=C4.1a|value=response'
# science_based_intensity_target: 'column=C4.1b_C17_Provide details of your emissions intensity target(s) and progress made against those target(s). - Is this a science-based target?|sheet=C4.1b|value=response'
# active_target: 'column=C4.1_Did you have an emissions target that was active in the reporting year?|sheet=C4 - Targets and Performance|value=response'
# active_target: 'column=9.2. Do you have a current emissions reduction target?|sheet=Strategy 9 (2)|value=response'
# active_target: 'column=CC3.1 - Did you have an emissions reduction or renewable energy consumption or production target that was active (ongoing or reached completion) in the reporting year?|sheet=CC3. Targets and Initiatives|value=response'
# (other climate-related targets, we never considered: 'column=C4.2_Did you have any other climate-related targets that were active in the reporting year?|sheet=C4 - Targets and Performance|value=response'
# scope1: 'column=C6.1_C1_What were your organization’s gross global Scope 1 emissions in metric tons CO2e? - Gross global Scope 1 emissions (metric tons CO2e)|sheet=C6.1|value=response'
# scope2: 'column=CC8.3. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e|sheet=CC8. Emissions Data|value=response'
# scope2_loc: 'column=C6.3_C1_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, location-based|sheet=C6.3|value=response'
# scope2_market: 'column=C6.3_C2_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, market-based (if applicable)|sheet=C6.3|value=response'
# scope3: 'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response'
# excluded_sources: 'column=C6.4a_C1_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Source|sheet=C6.4a|value=response',
# excluded_reason: 'column=C6.4a_C5_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Explain why this source is excluded|sheet=C6.4a|value=response'

['column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response',
 'column=C5.2a_Provide details of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response']

In [22]:
pd.tuple(df['column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response']).unique()

AttributeError: module 'pandas' has no attribute 'tuple'

In [71]:
# s1_verification: 'column=C10.1a_C3_Provide further details of the verification/assurance undertaken for your Scope 1 emissions, and attach the relevant statements. - Type of verification or assurance|sheet=C10.1a|value=response'


# initiative_details: 'column=C4.3b_C2_Provide details on the initiatives implemented in the reporting year in the table below. - Description of initiative|sheet=C4.3b|value=response'
# initiative_comment: 'column=C4.3b_C9_Provide details on the initiatives implemented in the reporting year in the table below. - Comment|sheet=C4.3b|value=response'
# change_dir_intensity: 'column=CC12.2 C6 - Please describe your gross global combined Scope 1 and 2 emissions for the reporting year in metric tonnes CO2e per unit currency total revenue - Direction of change from previous year|sheet=CC12.2|value=response'
# 'column=CC12.3. Direction of change from previous year|sheet=CC12.3|value=response' : 'change_dir'
# 'column=19.1. Do the absolute emissions (Scope 1 and Scope 2 combined) for the reporting year vary significantly compared to the previous year?|sheet=Emissions 18|value=response' : 'change_dir'
# 'column=C7.9a_C2_Identify the reasons for any change in your gross global emissions (Scope 1 and 2 combined), and for each of them specify how your emissions compare to the previous year. - Direction of change|sheet=C7.9a|value=response' : "change_dir_s12combined"

# 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type|sheet=C4.3b|value=response' :'initiative_type'
# 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response' : 'initiative_typeG'
# 'column=C4.3d_Why did you not have any emissions reduction initiatives active during the reporting year?|sheet=C4 - Targets and Performance|value=response': 'why_no_initiatives'
# 'column=C4.1a_C4_Provide details of your absolute emissions target(s) and progress made against those targets. - Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response': 'absolute_target_scope',
# 'column=C4.1b_C4_Provide details of your emissions intensity target(s) and progress made against those target(s). - Scope(s) (or Scope 3 category)|sheet=C4.1b|value=response': 'intensity_target_scope'
# 'column=CC3.1a C9 - Please provide details of your absolute target - Comment|sheet=CC3.1a|value=response': 'absolute_target_comment'
# 'column=CC3.1b C10 - Please provide details of your intensity target - Comment|sheet=CC3.1b|value=response': 'intensity_target_comment'

# 'column=C4.1a_C14_Provide details of your absolute emissions target(s) and progress made against those targets. - Is this a science-based target?|sheet=C4.1a|value=response': 'science_based_absolute_target',
# 'column=C4.1b_C17_Provide details of your emissions intensity target(s) and progress made against those target(s). - Is this a science-based target?|sheet=C4.1b|value=response': 'science_based_intensity_target'
# 'column=C4.1_Did you have an emissions target that was active in the reporting year?|sheet=C4 - Targets and Performance|value=response': 'active_target'
# 'column=9.2. Do you have a current emissions reduction target?|sheet=Strategy 9 (2)|value=response': 'active_target2'
# 'column=CC3.1 - Did you have an emissions reduction or renewable energy consumption or production target that was active (ongoing or reached completion) in the reporting year?|sheet=CC3. Targets and Initiatives|value=response': 'active_target3'
# (other climate-related targets, we never considered: 'column=C4.2_Did you have any other climate-related targets that were active in the reporting year?|sheet=C4 - Targets and Performance|value=response'
# 'column=C6.1_C1_What were your organization’s gross global Scope 1 emissions in metric tons CO2e? - Gross global Scope 1 emissions (metric tons CO2e)|sheet=C6.1|value=response': 'scope1'
# 'column=CC8.3. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e|sheet=CC8. Emissions Data|value=response': 'scope2',
# 'column=C6.3_C1_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, location-based|sheet=C6.3|value=response': 'scope2_loc',
# 'column=C6.3_C2_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, market-based (if applicable)|sheet=C6.3|value=response': 'scope2_market'
# 'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response': 'scope3',
# 'column=C6.4a_C1_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Source|sheet=C6.4a|value=response': 'excluded_sources',
# 'column=C6.4a_C5_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Explain why this source is excluded|sheet=C6.4a|value=response': 'excluded_reason'

# OK, going to try converting my R code base_KPI_labels.csv to python
to save conversion time of parquet to CSV, and CSV loading into R
and also because our code is going to probably all be in Python when we submit paper

In [23]:
df_core_survey.rename(columns={
    'column=Account number|sheet=*|value=response': 'account_no',
    'column=year|sheet=*|value=meta': 'year',
    'column=label|sheet=*|value=meta': 'respondent',
    'column=Organization|sheet=Summary Data|value=response': 'name',
    'column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response': 'protocol',
    'column=C5.2a_Provide details of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response': 'protocol_details',
    'column=C4.3b_C2_Provide details on the initiatives implemented in the reporting year in the table below. - Description of initiative|sheet=C4.3b|value=response': 'initiative_details',
    'column=C4.3b_C9_Provide details on the initiatives implemented in the reporting year in the table below. - Comment|sheet=C4.3b|value=response': 'initiative_comment',
    'column=CC12.3. Direction of change from previous year|sheet=CC12.3|value=response' : 'change_dir',
    'column=C7.9a_C2_Identify the reasons for any change in your gross global emissions (Scope 1 and 2 combined), and for each of them specify how your emissions compare to the previous year. - Direction of change|sheet=C7.9a|value=response' : "change_dir_s12combined",
    'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type|sheet=C4.3b|value=response' :'initiative_type',
    'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response' : 'initiative_typeG',
    'column=C4.3d_Why did you not have any emissions reduction initiatives active during the reporting year?|sheet=C4 - Targets and Performance|value=response': 'why_no_initiatives',
    'column=C4.1a_C4_Provide details of your absolute emissions target(s) and progress made against those targets. - Scope(s) (or Scope 3 category)|sheet=C4.1a|value=response': 'absolute_target_scope',
    'column=C4.1b_C4_Provide details of your emissions intensity target(s) and progress made against those target(s). - Scope(s) (or Scope 3 category)|sheet=C4.1b|value=response': 'intensity_target_scope',
    'column=CC3.1a C9 - Please provide details of your absolute target - Comment|sheet=CC3.1a|value=response': 'absolute_target_comment',    
    'column=CC3.1b C10 - Please provide details of your intensity target - Comment|sheet=CC3.1b|value=response': 'intensity_target_comment',
    'column=C4.1a_C14_Provide details of your absolute emissions target(s) and progress made against those targets. - Is this a science-based target?|sheet=C4.1a|value=response': 'science_based_absolute_target',
    'column=C4.1b_C17_Provide details of your emissions intensity target(s) and progress made against those target(s). - Is this a science-based target?|sheet=C4.1b|value=response': 'science_based_intensity_target',
    'column=C4.1_Did you have an emissions target that was active in the reporting year?|sheet=C4 - Targets and Performance|value=response': 'active_target1',
    'column=9.2. Do you have a current emissions reduction target?|sheet=Strategy 9 (2)|value=response': 'active_target2',
    'column=CC3.1 - Did you have an emissions reduction or renewable energy consumption or production target that was active (ongoing or reached completion) in the reporting year?|sheet=CC3. Targets and Initiatives|value=response': 'active_target3',
    'column=C6.1_C1_What were your organization’s gross global Scope 1 emissions in metric tons CO2e? - Gross global Scope 1 emissions (metric tons CO2e)|sheet=C6.1|value=response': 'scope1',
    'column=8.2b. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2b|value=response': 'scope1_cont',
    #'column=CC9.2e. Scope 1 emissions (metric tonnes CO2e)|sheet=CC9.2e|value=response': 'scope1_cont2',
    'column=CC8.3. Please provide your gross global Scope 2 emissions figures in metric tonnes CO2e|sheet=CC8. Emissions Data|value=response': 'scope2',
    #'column=8.3b. Gross global Scope 2 emissions (metric tonnes CO2e)|sheet=8.3b|value=response': 'scope2_cont',
    'column=CC10.2c. Scope 2 emissions (metric tonnes CO2e)|sheet=CC10.2c|value=response': 'scope2_cont2',
    'column=C6.3_C1_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, location-based|sheet=C6.3|value=response': 'scope2_loc',
    'column=C6.3_C2_What were your organization’s gross global Scope 2 emissions in metric tons CO2e? - Scope 2, market-based (if applicable)|sheet=C6.3|value=response': 'scope2_market',
    'column=C6.5_C2_Account for your organization’s gross global Scope 3 emissions, disclosing and explaining any exclusions. - Metric tonnes CO2e|sheet=C6.5|value=response': 'scope3',
    'column=C6.4a_C1_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Source|sheet=C6.4a|value=response': 'excluded_sources',
    'column=C6.4a_C5_Provide details of the sources of Scope 1 and Scope 2 emissions that are within your selected reporting boundary which are not included in your disclosure. - Explain why this source is excluded|sheet=C6.4a|value=response': 'excluded_reason'
}, inplace=True)

/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_83293/3180289502.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_core_survey.rename(columns={


In [25]:
df_core_survey_select_columns = df_core_survey[['account_no', 'year', 'name', 'respondent',
                                                'protocol', 'protocol_details',
               'initiative_details', 'initiative_comment', 
                'initiative_type', 'initiative_typeG', 
                'why_no_initiatives',
                'absolute_target_scope', 'intensity_target_scope',
                'absolute_target_comment', 'intensity_target_comment',
                'science_based_absolute_target', 'science_based_intensity_target',
                'active_target1', 'active_target2', 'active_target3',
                'scope1', 'scope1_cont', 
                'scope2', 'scope2_cont2', 
                'scope2_loc', 'scope2_market', 'scope3', 
                'excluded_sources', 'excluded_reason',
                'change_dir', 'change_dir_s12combined'
               ]]

# numbers
for colname in ['year', 'scope1', 'scope1_cont', 
                'scope2', 'scope2_cont2', 'scope2_loc', 'scope2_market', 'scope3']:

    df_core_survey_select_columns.loc[:,colname] = (
        df_core_survey_select_columns[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
        .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
    )
#     df.loc[:,'account_no'] = (
#         df['account_no']
#         .astype(str)
#         .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
#         .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
#     )

    

# strings

for colname in ['protocol', 'protocol_details', 
                'account_no', 'respondent', 'name', 'initiative_details', 'initiative_comment', 
                'initiative_type', 'initiative_typeG', 
                'why_no_initiatives',
                'absolute_target_scope', 'intensity_target_scope',
                'absolute_target_comment', 'intensity_target_comment',
                'science_based_absolute_target', 'science_based_intensity_target',
                'active_target1', 'active_target2', 'active_target3',
                'excluded_sources', 'excluded_reason',
                'change_dir', 'change_dir_s12combined']:

    df_core_survey_select_columns.loc[:,colname] = (
        df_core_survey_select_columns[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    )

In [26]:
df_core_survey_select_columns

,account_no,year,name,respondent,protocol,protocol_details,initiative_details,initiative_comment,initiative_type,initiative_typeG,...,scope1_cont,scope2,scope2_cont2,scope2_loc,scope2_market,scope3,excluded_sources,excluded_reason,change_dir,change_dir_s12combined
0,200,2010,Actelios SpA,investor,The Greenhouse Gas Protocol: A Corporate Accou...,None,None,None,None,None,...,NaN,12750.0,NaN,NaN,NaN,181.0,None,None,None,None
1,1800,2010,Bharat Petroleum Corporation,investor,India GHG Inventory Programme,We look into the possibility to adopt methodol...,None,None,None,None,...,NaN,225867.0,NaN,NaN,NaN,NaN,Diesel Fuel,Diesel purchased by Company owned vehicles are...,None,None
2,5300,2010,EDP - Energias de Portugal S.A.,investor,The Greenhouse Gas Protocol: A Corporate Accou...,EDP produces CO2 emissions in stationary combu...,None,None,None,None,...,NaN,1274421.0,NaN,NaN,NaN,NaN,None,None,None,None
3,29900,2010,Ernst & Young LLP UK,investor,The Greenhouse Gas Protocol: A Corporate Accou...,"""We capture raw data for all our UK based offi...",None,None,None,None,...,NaN,13234.0,NaN,NaN,NaN,19108.0,"""Fugitive fluorinated GHGs from refrigerant an...",We do not currently have a data collation proc...,None,None
4,28600,2010,Fortune Fashions Industries,investor,The Greenhouse Gas Protocol: A Corporate Accou...,http://www.epa.gov/cleanenergy/energy-resource...,None,None,None,None,...,NaN,4251.0,NaN,NaN,NaN,NaN,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40999,34421,2020,ZINWELL CORPORATION,supply_chain,China Corporate Energy Conservation and GHG Ma...,Question not applicable,None,1. Turn off the workstation lights where no pe...,"Other, please specify: Process emissions reduc...","Other, please specify Energy efficiency in bui...",...,NaN,NaN,NaN,NaN,NaN,NaN,Question not applicable,Question not applicable,None,Increased Increased Question not applicable\n ...
41000,838191,2020,ZKL PRINTING,supply_chain,ABI Energia Linee Guida; ISO 14064-1,Question not applicable,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,Question not applicable,Question not applicable,None,Question not applicable Question not applicabl...
41001,37896,2020,Zones,supply_chain,None,Question not applicable,None,"To reduce greenhouse gas emissions, Zones retr...","Heating, Ventilation and Air Conditioning (HVAC)",Energy efficiency in buildings,...,NaN,NaN,NaN,NaN,NaN,NaN,Question not applicable,Question not applicable,None,Question not applicable Question not applicabl...
41002,71988,2020,ZOTEQ,supply_chain,China Corporate Energy Conservation and GHG Ma...,Question not applicable,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,Question not applicable,Question not applicable,None,Question not applicable Question not applicabl...


In [30]:
df_core_survey_select_columns['protocol_details'].unique().tolist()

['None',
 'We look into the possibility to adopt methodologies set out in green house gas protocol i.e. WRI and WBCSD, for accounting in future years. (WRI – World Resource Institute, WBCSD – World Business Council for Sustainable Development).',
 'EDP produces CO2 emissions in stationary combustion (thermal power plants and gas pipeline stations) and mobile combustion, as well as SF6 emissions due to leakages of transformers, in power generation and distribution. EDP Scope 1 stationary combustion emissions were calculated using the methodology defined by the European Directive nº 2003/87/CE. This methodology calculates CO2 emissions from fuel consumption using measured data, emission factor and oxidation factor. EDP’s Scope 1 fleet emissions and Scope 2 emissions were calculated using the GHG protocol guidelines and EDP data. Both scopes are calculated in excel worksheets. Each EDP power plant in Portugal reports greenhouse gas emissions using the methodology approved by APA (Agência 

In [134]:
# are there any years that are entirely missing values of Scope 1 emissions?
print(df_core_survey_select_columns.groupby('year')['scope1'].apply(lambda x: x.notnull().any()))
print(df_core_survey_select_columns.groupby('year')['scope1_cont'].apply(lambda x: x.notnull().any()))

print(df_core_survey_select_columns.groupby('year')['scope2'].apply(lambda x: x.notnull().any()))
print(df_core_survey_select_columns.groupby('year')['scope2_loc'].apply(lambda x: x.notnull().any()))
print(df_core_survey_select_columns.groupby('year')['scope2_market'].apply(lambda x: x.notnull().any()))
print(df_core_survey_select_columns.groupby('year')['scope2_cont2'].apply(lambda x: x.notnull().any()))

(df['column=8.2d. Gross global Scope 1 emissions (metric tonnes CO2e)|sheet=8.2d|value=response']
 .astype(str)
 .str.replace(r"[\[\]']", '', regex=True)
 .pipe(pd.to_numeric, errors='coerce')
 .notnull().any()
)

year
2010     True
2011     True
2012     True
2013     True
2014     True
2015     True
2016     True
2017     True
2018    False
2019    False
2020    False
Name: scope1, dtype: bool
year
2010    False
2011    False
2012    False
2013    False
2014    False
2015    False
2016    False
2017    False
2018    False
2019    False
2020    False
Name: scope1_cont, dtype: bool
year
2010     True
2011     True
2012     True
2013     True
2014     True
2015     True
2016    False
2017    False
2018    False
2019    False
2020    False
Name: scope2, dtype: bool
year
2010    False
2011    False
2012    False
2013    False
2014    False
2015    False
2016     True
2017     True
2018    False
2019    False
2020    False
Name: scope2_loc, dtype: bool
year
2010    False
2011    False
2012    False
2013    False
2014    False
2015    False
2016     True
2017     True
2018    False
2019    False
2020    False
Name: scope2_market, dtype: bool
year
2010    False
2011     True
2012     True
2013     Tru

np.False_

In [157]:
account_nos = ['73924', '1808', '3558', '15366', '1680', '2382', '17770', '14581', '609', '31679', '58686', '14060', '9312', '21448', '12798', '31558', '19316', '4899', '351', '724', '17516', '19051', '1187', '60580', '53669', '50088', '44950', '3649', '21456', '2415', '11077', '14049', '20614', '4627', '50099', '14842', '8914', '13870', '13793', '6086', '20108', '15493', '31965', '31428', '31360', '35801', '53615', '9993', '32044', '22924', '10088', '16789', '8377', '22768', '10718', '46278', '23095', '22735', '9577', '31888', '21458', '44985', '3310', '3012', '3642', '10791', '3934', '5047', '1405', '37237', '14065', '20058', '17372', '1650', '18122', '18377', '3672', '8525', '37535', '832133', '12340', '7013', '15288', '2392', '53635', '68215', '53642', '36686', '7119', '74136', '51121', '20280', '18141', '3563', '21889', '21677', '51312', '37067', '3249', '28835', '3182', '17114', '14653', '74082', '73479', '3169', '45029', '37482', '37464', '37469', '49557', '51307', '22698', '52872', '51307', '32055', '32634', '49530', '34024', '32634', '3082', '68181', '31876', '625', '4674', '403', '5566', '3599', '21380', '50078', '4645', '15345', '4647', '4706', '53647', '18301', '22175', '4902', '6688', '74137', '593', '9044', '6795', '8207', '13429', '8082', '22369', '30215', '4630', '9915', '9548', '74048', '19227', '4312', '6555', '15911', '21707', '6534', '16109', '31923', '21355', '4081', '57666', '9031', '5584', '7084', '3022', '53', '7788', '814', '14112', '23145', '20408', '569', '51086', '23142', '14106', '10254', '23130', '49671', '6582', '22539', '458', '68198', '13167', '22032', '19789', '53776', '14805', '10609', '4050', '6535', '5802', '10205', '17320', '53754', '2880', '7139', '21734', '51285', '16563', '1062', '49590', '21722', '3545', '14930', '11063', '173', '21773', '21769', '21744', '5921', '21751', '2096', '8741', '7808', '53746', '12874', '9101', '9423', '15675', '3515', '9810', '16722', '7562', '16672', '12389', '31452', '19402', '10649', '1344', '15662', '38132', '44457', '18354', '19402', '3042', '68399', '31463', '17628', '19829', '16130', '8031', '22156', '1823', '13146', '9112', '2414', '820', '19402', '15875', '17301', '8097', '19834', '12828', '17624', '9283', '36028', '22100', '19856', '7677', '12389', '58707', '4835', '11328', '20146', '22799', '4377', '21811', '2097', '53006', '15521', '10781', '4059', '22970', '23046', '16760', '8657', '58741', '10629', '68167', '53683', '6806', '22142', '15282', '772', '17301', '11944', '16630', '22257', '20606', '6032', '23214', '12376', '19402', '36028', '14526', '17565', '4575', '14637', '23011', '19589', '10871', '10649', '7702', '2297', '53683', '20548', '15971', '4730', '10055', '10453', '22163', '35739', '20799', '15283', '31448', '15981', '35739', '20548', '56376', '15971', '17190', '10156', '9359', '11638', '8884', '58741', '44665', '573', '8187', '8057', '3527', '29530', '12552', '3227', '55153', '3229', '18047', '8058', '3503', '22766', '11267', '11353', '7014', '143', '8370', '19631', '9025', '9026', '1800', '22188', '63519', '32014', '47918', '668', '18286', '18285', '9834', '31998', '54167', '1321', '10295', '32000', '12603', '19110', '8361', '22352', '1411', '600', '8765', '9673', '53814', '53814', '2520', '10162', '21991', '8892', '1134', '31285', '22206', '31544', '18579', '87', '36752', '53604', '17279', '9363', '8254', '31273', '6331', '22221', '19794', '5574', '313', '31290', '5804', '11640', '68188', '31289', '31321', '22536', '22536', '22536', '22536', '9960', '57975', '520', '9703', '50182', '19792', '10754', '22242', '19559', '12631', '9520', '31908', '15365', '16399', '13352', '18571', '9641', '20926', '19190', '9134', '18156', '16728', '10755', '12913', '17899', '6161', '54991', '17918', '12303', '8042', '20081', '830', '55086', '5145', '11520', '10179', '20597', '8338', '18297', '17897', '20878', '1046', '12414', '18292', '18266', '246', '3889', '9992', '3402', '6193', '13367', '12291', '19266', '3316', '12120', '20906', '50241', '13317', '20912', '992', '16852', '12086', '37310', '37906', '6855', '4223', '19195', '53586', '10234', '22228', '22241', '17315', '45081', '51172', '8373', '58901', '19184', '19254', '4230', '18170', '10296', '13321', '32738', '9863', '6669', '59462', '54751', '8566', '31907', '16601', '16301', '17895', '13575', '18316', '58907', '54767', '31538', '13909', '12263', '18074', '50183', '9712', '7869', '4255', '22249', '15485', '37360', '57929', '16876', '258', '4268', '4256', '3306', '58912', '13273', '22197', '8332', '13363', '9538', '9972', '21327', '15693', '10177', '51663', '13316', '22274', '47029', '12271', '10135', '17313', '68310', '16294', '45167', '19163', '1681', '8601', '42029', '16650', '3312', '49535', '22264', '8032', '11413', '4237', '19164', '10379', '19152', '56044', '261', '722', '7843', '17073', '50064', '10338', '16168', '4861', '21673', '10076', '16186', '49955', '59917', '21207', '4192', '22507', '44834', '10673', '16891', '8719', '22978', '9979', '44839', '10668', '9050', '8696', '47244', '10227', '10674', '10268', '16188', '8708', '36933', '22508', '20776', '4190', '21216', '31746', '22713', '49596', '20332', '31854', '11289', '21419', '13279', '838479', '828251', '62138', '56592', '1941', '837690', '837062', '56848', '836287', '30813', '40219', '34391', '837872', '830437', '62288', '61443', '59271', '57434', '826968', '828565', '5858', '830244', '46939', '56372', '18405', '39362', '827228', '1356', '828245', '62916', '828261', '34392', '40342', '123', '837443', '61015', '837117', '838148', '73380', '38459', '52262', '61090', '57265', '57411', '52425', '50828', '31823', '34327', '60780', '73397', '71825', '837947', '839872', '9037', '63570', '834602', '15373', '15279', '30039', '830898', '57157', '51284', '46940', '839797', '37009', '829307', '837237', '46679', '837702', '57553', '15093', '41902', '827158', '16585', '61317', '38138', '1124', '73406', '71563', '7540', '839176', '56666', '828375', '61104', '828416', '39066', '56059', '51834', '61980', '19802', '1408', '827002', '63540', '829475', '828407', '55988', '72122', '836850', '830475', '12335', '33515', '52480', '838136', '51987', '57255', '827547', '70037', '63454', '40461', '54744', '52854', '828078', '9301', '56534', '51166', '2595', '70469', '56055', '57397', '16923', '56304', '840057', '70427', '838400', '72120', '828402', '30591', '48503', '838449', '18421', '838447', '57320', '71124', '10733', '39421', '8237', '56576', '19439', '836660', '47056', '7904', '10494', '828421', '53580', '61802', '59905', '34079', '29969', '831830', '838203', '61869', '30100', '32895', '72083', '838338', '48098', '35761', '55662', '71105', '838175', '827983', '2091', '828110', '51666', '56527', '23202', '56280', '2573', '30114', '70417', '839507', '8167', '72876', '6419', '40952', '30125', '828893', '39226', '5357', '71076', '57462', '32739', '829232', '11017', '22991', '840480', '837956', '15625', '47549', '829189', '840396', '32569', '3537', '829001', '31742', '72084', '37902', '40303', '1152', '71811', '838280', '830278', '828422', '62293', '35313', '839066', '3005', '586', '56596', '47758', '33253', '838191', '20159', '828417', '47736', '827295', '838287', '71138', '47249', '61695', '61315', '828105', '51245', '839531', '57290', '52389', '14169', '779', '44407', '62017', '19377', '57619', '51131', '57237', '837231', '32524', '61680', '61138', '830537', '829200', '839062', '838222', '16529', '61966', '836115', '61355', '31393', '22416', '5844', '50507', '50955', '62143', '13314', '10117', '837147', '55620', '33274', '70344', '61661', '60785', '838103', '829896', '57562', '51815', '831176', '827442', '838517', '46842', '72062', '829479', '3017', '15916', '840352', '828352', '17666', '829185', '34289', '828083', '51994', '19075', '44710', '839218', '839448', '70085', '61174', '828271', '32786', '8051', '828551', '60834', '51315', '41211', '57031', '839526', '57386', '840379', '10195', '5337', '11411', '827613', '37816', '38691', '58554', '826982', '51799', '2670', '837212', '837778', '41784', '9975', '62281', '32780', '838979', '831183', '839076', '828398', '15583', '36965', '51872', '828406', '13483', '1602', '38063', '837539', '38125', '16012', '61422', '1458', '828399', '56207', '35050', '834856', '839199', '838218', '831996', '61298', '70249', '60960', '51673', '12799', '61369', '840365', '839511', '838184', '71543', '4365', '22873', '56669', '45114', '5885', '9759', '20384', '839555', '70230', '16803', '56217', '14697', '828883', '1830', '33272', '840802', '57533', '40594', '827568', '2437', '52648', '828445', '829881', '6430', '828564', '828307', '837675', '46659', '46768', '51271', '838409', '838061', '97', '829903', '70639', '61181', '10498', '829442', '52166', '71255', '70134', '73315', '365', '829414', '524', '31585', '61042', '70031', '834853', '29059', '837713', '839207', '828652', '837960', '10612', '62058', '8675', '840355', '32134', '831677', '19201', '838518', '19328', '14654', '58212', '51128', '51505', '16114', '7690', '72427', '71090', '10696', '47759', '828879', '839748', '34448', '54554', '11904', '839240', '5207', '22874', '56201', '828380', '13813', '41279', '837077', '72088', '15169', '828304', '34382', '828866', '22331', '15831', '837044', '838119', '22734', '720', '70943', '1470', '838300', '72553', '62929', '51748', '33411', '51887', '35097', '71209', '3387', '20402', '50021', '828248', '35501', '47034', '4562', '71941', '40613', '56107', '837861', '119', '51658', '839315', '839111', '20801', '70033', '22082', '12781', '828868', '56801', '56167', '12889', '23612', '3349', '41429', '70645', '71145', '30047', '33541', '838144', '838432', '829921', '830234', '17604', '840491', '837236', '53510', '35790', '838087', '53106', '837113', '47540', '829025', '73207', '71689', '840059', '30498', '831913', '827215', '55796', '57478', '71115', '39231', '56776', '828513', '838039', '840272', '837767', '827486', '40271', '5229', '837066', '31902', '41214', '51510', '837169', '70020', '57761', '70022', '838564', '834706', '33261', '56934', '62318', '50878', '838186', '20705', '46978', '38007', '70407', '38092', '51806', '40310', '30086', '40989', '4911', '837867', '1693', '829334', '827482', '828544', '61565', '18169', '838455', '56624', '829129', '57479', '30440', '72477', '837870', '2986', '826921', '830294', '10216', '838977', '4058', '54554', '15623', '70616', '71194', '830534', '837461', '40659', '47067', '15046', '14061', '837837', '15980', '40781', '7538', '34494', '830419', '51409', '828326', '37994', '838109', '16606', '837881', '55612', '60941', '51141', '836841', '62316', '5519', '13873', '830210', '70630', '34728', '38537', '44321', '837963', '44674', '826997', '827275', '31492', '51485', '37177', '830373', '950', '70557', '32795', '12850', '7785', '20516', '57233', '23100', '30979', '56750', '52734', '72137', '51736', '35092', '837952', '838381', '52562', '62035', '51782', '38432', '70160', '828792', '10793', '56161', '30117', '71224', '57526', '836571', '59901', '53518', '840513', '838134', '839281', '34084', '57173', '827503', '839785', '837897', '839375', '769', '51294', '58304', '839875', '840250', '20869', '44', '828415', '14548', '72686', '57567', '34556', '71053', '60979', '31270', '40328', '1597', '53037', '70315', '51747', '47372', '53513', '50507', '837189', '28588', '44972', '73575', '57581', '57372', '29054', '3637', '47122', '30110', '830201', '840658', '5195', '836130', '828648', '29787', '2825', '20398', '33438', '47030', '830770', '48143', '70233', '840498', '46657', '1787', '52820', '826828', '70176', '57931', '35769', '836766', '839412', '832006', '826351', '40768', '4830', '4678', '829510', '834714', '838220', '58313', '831441', '829186', '64657', '17069', '828699', '9847', '51619', '54975', '837208', '29612', '838407', '56743', '6083', '47019', '51738', '828069', '6532', '55974', '20813', '840055', '829773', '55608', '839184', '40581', '839033', '837789', '16902', '34301', '61173', '46633', '44763', '837852', '828885', '9630', '38168', '831912', '47853', '48144', '33108', '56824', '57271', '840869', '838145', '71095', '47515', '829441', '55992', '50818', '838561', '62319', '3779', '8880', '840063', '35024', '5414', '836865', '46977', '61472', '52369', '836241', '35322', '831417', '16804', '62296', '62121', '52701', '838519', '828353', '826825', '56558', '15097', '61984', '827444', '828290', '52983', '20896', '47254', '70279', '826931', '34439', '18095', '34395', '62024', '33293', '1271', '52154', '838113', '70619', '53675', '71227', '62059', '56078', '836813', '832323', '58651', '55813', '12382', '58857', '39408', '56526', '19376', '8587', '18585', '837188', '828696', '840635', '917', '62954', '836237', '56846', '827485', '837473', '72428', '22367', '5624', '1179', '47404', '71199', '56720', '60947', '39684', '831636', '829697', '830597', '838086', '61299', '34421', '837855', '31771', '828967', '62656', '70147', '56505', '21141', '52034', '71717', '827414', '48125', '10395', '4271', '830760', '828319', '5087', '28844', '61696', '6361', '35762', '827013', '71107', '839537', '57238', '57190', '40649', '71285', '52918', '52279', '830759', '55567', '55754', '839193', '34462', '52682', '61408', '33767', '61794', '6333', '5377', '3922', '47483', '628', '61563', '837154', '827537', '33481', '40300', '838302', '827287', '32296', '55580', '829128', '61137', '828419', '828571', '50820', '828119', '37382', '34340', '828112', '1219', '828000', '837515', '36807', '70419', '13439', '40288', '28953', '836327', '837938', '14896', '13562', '62883', '47737', '60620', '61709', '831175', '840353', '41491', '45142', '829529', '830288', '61981', '837078', '1591', '838174', '21641', '56524', '57503', '840487', '15306', '39764', '56037', '836319', '52338', '830273', '840623', '839192', '55607', '61180', '23504', '57278', '834837', '71709', '829760', '51199', '840496', '55576', '837816', '838403', '31070', '840748', '55980', '60779', '17684', '58641', '3583', '837714', '828490', '838539', '779', '61723', '72711', '48852', '836209', '62298', '829401', '827452', '63499', '827220', '837244', '830168', '829847', '73608', '71788', '829779', '10432', '10217', '63525', '828328', '51122', '837957', '1884', '57536', '7345', '41522', '830382', '53080', '4853', '4829', '838487', '8261', '47919', '827180', '57343', '73394', '838501', '57495', '10175', '33636', '47818', '837598', '6332', '2360', '58002', '61824', '72066', '839722', '828281', '830250', '828420', '14667', '30766', '34497', '839894', '61706', '21782', '22859', '22615', '699', '3358', '28797', '839624', '12860', '15509', '31303', '51676', '827863', '828366', '41714', '28826', '33597', '68200', '51425', '70601', '46848', '38032', '828667', '70613', '838572', '829463', '70401', '839059', '5767', '57246', '827267', '56081', '829527', '38154', '73401', '3848', '14089', '55137', '827102', '28625', '11427', '40779', '72680', '838192', '829396', '35831', '829884', '839219', '838334', '56291', '46599', '14961', '44710', '828339', '827991', '838994', '828452', '828093', '46851', '828286', '1536', '829372', '33701', '47784', '61989', '8580', '839251', '829947', '56396', '61381', '11853', '20917', '71627', '56021', '56296', '70139', '836823', '40672', '39166', '52680', '51804', '827524', '20111', '837712', '70152', '71214', '11655', '838327', '56212', '40560', '827206', '62044', '830303', '57225', '838492', '3694', '71634', '32144', '62149', '57878', '61669', '828098', '39716', '9310', '49682', '51778', '828404', '55832', '829451', '829551', '839920', '33374', '34322', '838992', '33739', '41575', '838313', '61392', '829013', '47239', '57321', '14820', '1417', '828135', '71975', '827191', '10882', '51843', '836231', '9881', '20360', '827151', '1193', '838312', '72052', '51288', '61078', '71724', '61560', '837108', '830274', '837759', '52088', '37764', '71168', '10057', '20936', '57625', '838181', '40961', '70510', '13042', '35814', '4408', '2695', '32572', '51395', '18320', '38019', '829123', '62672', '52398', '837271', '39725', '59467', '836136', '58251', '837587', '71192', '10143', '838396', '828072', '830120', '839775', '828342', '32733', '70437', '1951', '59396', '47212', '826955', '70212', '827978', '33788', '73480', '61832', '70587', '28840', '73490', '837287', '57212', '9119', '29188', '29952', '830700', '829880', '71178', '826960', '4291', '38359', '33452', '829965', '8054', '59134', '57461', '829484', '51397', '838763', '29052', '29524', '838718', '34512', '829741', '71708', '52363', '61314', '34291', '59242', '21481', '36604', '7271', '61689', '36702', '837096', '828099', '837194', '836310', '38819', '840056', '11085', '829554', '829910', '60975', '47400', '56831', '20055', '826935', '838451', '2095', '71058', '71957', '61686', '56294', '34082', '71129', '830261', '3270', '5197', '61400', '3133', '839063', '28560', '61186', '839400', '839548', '56575', '836814', '23521', '70423', '839108', '70086', '47069', '36860', '51718', '1318', '707', '51792', '839274', '40413', '48881', '12374', '7286', '829366', '837543', '2836', '830194', '13117', '830239', '828359', '838040', '827108', '71097', '15422', '15265', '33899', '1198', '32134', '828394', '60917', '14030', '70158', '51827', '72165', '61198', '33259', '47510', '62076', '834702', '837755', '829188', '829448', '58', '2942', '22797', '47255', '70637', '57557', '51638', '38167', '838726', '57276', '51146', '56048', '56056', '13074', '28853', '50905', '57542', '72193', '33161', '60944', '831633', '45126', '46702', '6285', '34308', '838084', '839615', '827972', '52519', '39747', '15271', '51878', '63', '836840', '30724', '827113', '71896', '57472', '830372', '838978', '51311', '827921', '828312', '70027', '362', '56583', '32748', '826993', '838276', '52177', '38112', '51656', '838047', '33241', '41697', '829007', '71120', '661', '838170', '20173', '34437', '19393', '828401', '73513', '73409', '61660', '57422', '17842', '828451', '830222', '837604', '839826', '16102', '828349', '62312', '839000', '56549', '38340', '836109', '838806', '71111', '30634', '745', '23281', '62905', '70974', '827610', '582', '34081', '28973', '49682', '61308', '837143', '838414', '62862', '62211', '839208', '38068', '836883', '827203', '829486', '23217', '831828', '57427', '833291', '828838', '8526', '56727', '839265', '28800', '828137', '29501', '831161', '828361', '56305', '828068', '70019', '41374', '3640', '830589', '839790', '840616', '52169', '55558', '13579', '838513', '61178', '62300', '72147', '30586', '52828', '35800', '21791', '837849', '57266', '47202', '56811', '350', '33715', '838351', '829223', '827270', '35778', '28949', '829193', '838972', '15072', '828400', '41827', '55601', '51598', '10405', '51157', '57467', '17929', '838072', '57241', '62055', '826598', '71637', '46769', '839231', '56877', '61544', '15949', '51171', '60857', '838416', '828116', '56058', '28869', '828468', '827169', '28842', '56903', '55989', '52378', '838986', '14266', '38107', '10233', '6414', '838383', '57570', '73263', '47553', '839258', '838056', '41907', '70105', '829379', '36842', '831675', '10797', '70982', '51746', '837665', '60808', '838976', '73408', '828231', '51611', '62314', '837466', '14783', '32929', '72723', '30092', '51228', '63625', '40742', '47245', '56808', '836386', '33836', '38000', '828874', '828122', '56939', '5198', '21491', '837242', '56203', '61030', '13627', '61423', '828126', '57451', '50310', '828872', '51742', '40574', '830793', '827445', '57504', '70624', '837693', '57236', '16407', '837103', '14916', '40316', '837172', '828088', '38126', '840062', '62158', '839234', '61037', '47916', '837873', '839414', '827518', '58300', '50950', '50155', '828412', '33496', '828871', '23128', '2906', '70228', '53090', '46847', '15829', '838974', '829423', '55568', '20851', '32788', '837440', '829444', '828438', '702', '56114', '33628', '829927', '48565', '828701', '39219', '61688', '828131', '47100', '33185', '826893', '2667', '18553', '72124', '838340', '830271', '2841', '71217', '829211', '28924', '62903', '72110', '1213', '838140', '40676', '57551', '63537', '828575', '33890', '597', '836187', '56887', '837891', '40719', '838392', '9578', '827625', '829370', '29617', '44650', '7060', '828138', '826932', '56800', '838490', '737', '38118', '61305', '831869', '61823', '837860', '52397', '838177', '46817', '828555', '839759', '837149', '61872', '10124', '840012', '73398', '29751', '34144', '41240', '74263', '56447', '38980', '837631', '827650', '51909', '73385', '51895', '38175', '838452', '54747', '1653', '1481', '838044', '836233', '13400', '72430', '47166', '13649', '831911', '827966', '40611', '51072', '52246', '71836', '38043', '70429', '7540', '19083', '57836', '828288', '838576', '836137', '840654', '56935', '14855', '828134', '837198', '72841', '37856', '33882', '56958', '57540', '838457', '22199', '52224', '46726', '10294', '838157', '52587', '73347', '834850', '33461', '70659', '61555', '829957', '38124', '827208', '19084', '834496', '631', '62278', '72031', '716', '29855', '57459', '828378', '828005', '72135', '52935', '53607', '828052', '828360', '57419', '839371', '56997', '56816', '57455', '62067', '838171', '61821', '33183', '56402', '4816', '62861', '46958', '34612', '838379', '827719', '838180', '56282', '61306', '837649', '837402', '828430', '71698', '71807', '62079', '71718', '58646', '14910', '56546', '61033', '4832', '44674', '839246', '1146', '829422', '55570', '829846', '3946', '828365', '55137', '72474', '52476', '19145', '62305', '830296', '6684', '71865', '834785', '17180', '61024', '51430', '838166', '28632', '52914', '4638', '837772', '828405', '71180', '71860', '33809', '57543', '50924', '839829', '51745', '33437', '55595', '838179', '40536', '17421', '829489', '57486', '61318', '830251', '149', '12273', '52236', '836424', '37110', '837249', '51196', '834701', '57281', '16531', '57435', '829939', '838317', '828002', '839451', '40889', '15419', '23106', '829184', '16700', '57256', '9327', '38108', '61192', '18448', '38022', '56175', '58316', '838057', '40970', '838680', '6383', '56783', '41248', '52385', '57289', '826375', '61310', '19241', '55559', '33870', '34446', '33677', '829709', '73550', '56728', '52170', '34451', '2861', '827506', '828878', '70098', '2588', '51689', '47361', '57370', '837125', '47837', '72002', '119', '57303', '831827', '28988', '70433', '19366', '71144', '51864', '838046', '35063', '57230', '17815', '839120', '62297', '70096', '14901', '57534', '3010', '8838', '252', '827306', '22383', '839534', '838190', '56145', '61705', '836945', '33173', '840622', '35323', '828877', '828882', '51685', '61311', '902', '71641', '57259', '38008', '47899', '51120', '40872', '5540', '57179', '839871', '2191', '71691', '839004', '837655', '51150', '828442', '40966', '34388', '28692', '829517', '73516', '837728', '70658', '50895', '840481', '838886', '839509', '29900', '841157', '38024', '828553', '70594', '35086', '837723', '55588', '55962', '40195', '8634', '827039', '56933', '3751', '807', '47578', '53060', '3062', '838481', '32914', '840655', '51681', '828374', '47487', '827514', '34348', '13532', '827037', '17063', '838776', '62148', '4652', '3944', '38207', '47448', '56165', '828262', '51642', '839499', '837232', '37851', '836846', '10109', '38005', '41917', '828001', '19304', '830415', '828816', '31484', '14928', '29505', '34951', '13459', '836103', '828444', '11141', '17937', '827454', '36602', '831427', '8670', '61711', '38144', '837250', '832077', '51278', '60798', '2455', '56741', '830184', '838123', '838524', '828363', '839773', '58642', '828106', '29745', '840252', '837115', '56076', '828448', '22370', '828269', '71319', '839129', '52335', '21063', '838809', '4428', '829700', '47909', '837642', '839484', '61375', '51488', '831358', '38026', '35262', '53011', '13489', '22372', '47752', '830116', '838401', '73316', '829335', '63535', '840733', '56223', '14830', '71840', '51451', '827205', '38059', '838328', '40390', '72651', '13849', '57569', '51297', '291', '56166', '828920', '828244', '828431', '70644', '838211', '828573', '71195', '9411', '41399', '838326', '40442', '2044', '828127', '46807', '839799', '831540', '8027', '830188', '41274', '41021', '827404', '71667', '838298', '834854', '72571', '38829', '829478', '828109', '836967', '838310', '828795', '28915', '838339', '20771', '13082', '837278', '19271', '29744', '56300', '28808', '834778', '20612', '62217', '838185', '33766', '828301', '52681', '1271', '57550', '71134', '838187', '73381', '64076', '30853', '56444', '30607', '62139', '39565', '63741', '55922', '839514', '52920', '839812', '828311', '71190', '837939', '70646', '56763', '57520', '56774', '34482', '828454', '11118', '31648', '30056', '7105', '36701', '828090', '830780', '11578', '56043', '839045', '33309', '6685', '57326', '38592', '56844', '71264', '828495', '70203', '55939', '837552', '840469', '836762', '55711', '827509', '839629', '8274', '391', '73407', '16725', '58249', '839302', '836908', '836238', '831040', '838038', '30867', '837718', '828367', '830321', '839824', '836839', '62315', '839452', '3635', '22540', '70225', '70564', '70631', '11581', '837945', '56780', '827021', '838437', '837817', '61316', '52633', '59722', '40435', '49247', '18573', '3989', '51287', '22216', '60932', '47906', '41029', '73350', '56051', '71122', '837681', '51237', '839697', '52855', '51661', '72040', '837924', '828257', '10095', '39735', '70652', '827494', '832061', '826371', '837164', '51444', '38434', '41009', '836111', '837127', '35090', '55032', '832279', '22287', '59325', '44716', '832279', '21338', '22287', '68135', '37115', '8212', '16398', '135', '59325', '49607', '23102', '8212', '37115', '37134', '16398', '49607', '53612', '36845', '16137', '17519', '64654', '58780', '21306', '16525', '64656', '18650', '14649', '21537', '59170', '4810', '20149', '13496', '31267', '17834', '457', '23132', '64651', '73893', '50100', '6432', '23124', '6546', '15490', '3814', '19270', '20416', '1164', '10194', '32067', '21836', '6389', '134', '31611', '13838', '13838', '5300', '1397', '14877', '23270', '9747', '53553', '53553', '15957', '11043', '22844', '37329', '13542', '45228', '5339', '6772', '3420', '31468', '9054', '31467', '31144', '54927', '16365', '58768', '18061', '13392', '3450', '11049', '16406', '9785', '74103', '7903', '14516', '31623', '74102', '2913', '54203', '31652', '59175', '22379', '5346', '1151', '59172', '22298', '41022', '37091', '22631', '74155', '15294', '15297', '62844', '21129', '21143', '49585', '21112', '31678', '21150', '31759', '21122', '21160', '23633', '21158', '49618', '21145', '21115', '21159', '41437', '31565', '21131', '35233', '21154', '21126', '21128', '21144', '21134', '31563', '21151', '449', '31761', '6612', '18277', '22589', '22659', '47141', '12128', '3705', '47208', '21686', '74150', '3204', '21868', '34425', '4122', '6829', '31326', '21683', '7583', '22694', '55097', '47144', '21694', '34509', '21868', '22977', '7331', '21950', '895', '21925', '73955', '41539', '22603', '22566', '47043', '1037', '48068', '51198', '47036', '777', '19102', '8825', '22909', '31343', '15698', '17889', '31343', '13060', '1529', '19565', '22902', '23293', '14814', '23292', '31545', '4787', '18949', '10650', '138', '7776', '37113', '32507', '6147', '11860', '20773', '8688', '13019', '31341', '15653', '15578', '18436', '16848', '8104', '9865', '8528', '777', '6436', '12915', 
               '14766', '5532', '9865', '58673', '10350', '8881', '12622', '20754', '22908', '10350', '12546', '7791', '17637']
account_nos_str = [str(x) for x in account_nos]




df_core_survey_select_columns[df_core_survey_select_columns['account_no'].astype(str).isin(account_nos_str) & 
                              (df_core_survey_select_columns['year'] == 2019)]


df_core_survey_select_columns[df_core_survey_select_columns['account_no'].astype(str).isin(account_nos_str) & 
                              (df_core_survey_select_columns['year'] == 2019) &
                              (df_core_survey_select_columns['respondent'] == 'investor')
                             ]['account_no'] # 1137 investor, 1844 supply_chain

30728       44
30732       87
30733       97
30737      291
30738       53
         ...  
32732    53647
32736    73479
32738    21063
32739    41437
32740    31761
Name: account_no, Length: 1137, dtype: object

In [162]:
print(df_core_survey_select_columns[df_core_survey_select_columns['account_no'].astype(str).isin(['73924','31679','609','14581','17770', '2382', '1680','15366']) & 
                              (df_core_survey_select_columns['year'] == 2019) &
                              (df_core_survey_select_columns['respondent'] == 'investor')
                             ]['account_no'].tolist()) # 1137 investor-only


['609', '1680', '73924', '2382', '14581', '15366', '31679', '17770']


In [146]:
account_nos = ['16137']
account_nos_str = [str(x) for x in account_nos]





df_core_survey_select_columns[df_core_survey_select_columns['account_no'].astype(str).isin(account_nos_str) & 
                              (df_core_survey_select_columns['year'] == 2019)]


,account_no,year,name,respondent,initiative_details,initiative_comment,initiative_type,initiative_typeG,why_no_initiatives,absolute_target_scope,...,scope1_cont,scope2,scope2_cont2,scope2_loc,scope2_market,scope3,excluded_sources,excluded_reason,change_dir,change_dir_s12combined
32251,16137,2019,Salmar ASA,investor,Hydro Hydro,SalMar Farming has a project underway to run e...,Low-carbon energy installation Low-carbon ener...,None,Question not applicable,Scope 2 (location-based) Scope 3: Waste genera...,...,NaN,NaN,NaN,NaN,NaN,NaN,Question not applicable,Question not applicable,None,No change Decreased No change No change No cha...


In [142]:
type(account_nos_str)
type(df['account_no'])

pandas.core.series.Series

# continue with KPIs labels

In [74]:
df.rename(columns={
    'column=Account number|sheet=*|value=response': 'account_no',
    'column=year|sheet=*|value=meta': 'year',
    'column=label|sheet=*|value=meta': 'respondent',
    'column=Organization|sheet=Summary Data|value=response': 'name'
}, inplace=True) # the dataframe as it is, not make a copy to overflow memory!

In [10]:
# numbers

for colname in ['year', 'account_no']:

    df.loc[:,colname] = (
        df[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
        .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
    )
#     df.loc[:,'account_no'] = (
#         df['account_no']
#         .astype(str)
#         .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
#         .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
#     )

    

# strings

for colname in ['respondent']:

    df.loc[:,colname] = (
        df[colname]
        .astype(str)
        .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    )

In [11]:
df.head()

,year,respondent,name,account_no,column=Discloser ID|sheet=Summary|value=response,column=Country|sheet=Summary Data|value=response,column=Primary industry|sheet=Summary Data|value=response,column=Primary Expansion|sheet=Summary|value=response,column=secondary_expansion|sheet=Summary Data|value=response,column=Complexity|sheet=Summary|value=response,...,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=number_responses",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_characters,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_responses,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_characters,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_responses,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|model=climate_specificity|label=spec|value=score_average,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_characters,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_responses
0,2010,investor,[Actelios SpA],200.0,[f31d5885-e21b-df11-b692-0017a47708d8],[Italy],[Electric Utilities & Independent Power Produc...,[Electric Utilities 250],[None],[Long],...,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0
1,2010,investor,[Bharat Petroleum Corporation],1800.0,[a7245885-e21b-df11-b692-0017a47708d8],[India],[Oil and Gas],[India 200],[Emerging Markets 800],[Long],...,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0
2,2010,investor,[EDP - Energias de Portugal S.A.],5300.0,[b76c508b-e21b-df11-b692-0017a47708d8],[Portugal],[Electric Utilities & Independent Power Produc...,[Portugal 40],[Electric Utilities 250; Global 500; Euro 300;...,[Long],...,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0
3,2010,investor,[Ernst & Young LLP UK],29900.0,[986e2612-9261-df11-bb08-0017a47708d8],[United Kingdom],"[Banks, Diverse Financials, and Insurance]",[Mayday],[None],[Long],...,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0
4,2010,investor,[Fortune Fashions Industries],28600.0,[450e48b0-e640-df11-9c11-0017a47708d8],[USA],"[Textiles, Apparel, Footwear and Luxury Goods]",[Walmart],[None],[Long],...,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0


In [12]:
base2 = df[['account_no', 'year', 'respondent']]
base2.loc[:,'year'] = (
    base2['year']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'account_no'] = (
    base2['account_no']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'respondent'] = (
    base2['respondent']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
)

In [13]:
base2.head()

,account_no,year,respondent
0,200.0,2010,investor
1,1800.0,2010,investor
2,5300.0,2010,investor
3,29900.0,2010,investor
4,28600.0,2010,investor


### finding max cols filled by year

In [14]:
num_filled_cols = [col for col in df.columns if "number_responses" in col]
year_col = 'year'
calc = df[[year_col] + num_filled_cols] 

In [15]:
calc.loc[:,'year'] = (
    calc['year']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)

In [16]:
calc.head()

,year,column=Organization|sheet=Summary Data|value=number_responses,column=Account number|sheet=*|value=number_responses,column=Discloser ID|sheet=Summary|value=number_responses,column=Country|sheet=Summary Data|value=number_responses,column=Primary industry|sheet=Summary Data|value=number_responses,column=Primary Expansion|sheet=Summary|value=number_responses,column=secondary_expansion|sheet=Summary Data|value=number_responses,column=Complexity|sheet=Summary|value=number_responses,column=response_status|sheet=Summary Data|value=number_responses,...,"column=C4.2b_C6_Provide details of any other climate-related targets, including methane reduction targets. - Target denominator (intensity targets only)|sheet=C4.2b|value=number_responses","column=C4.2b_C11_Provide details of any other climate-related targets, including methane reduction targets. - Figure or percentage in reporting year|sheet=C4.2b|value=number_responses","column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|value=number_responses",column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=number_responses,column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|value=number_responses,column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|value=number_responses,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=number_responses",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_responses,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_responses,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_responses
0,2010,1,1,1,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2010,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,2010,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,2010,1,1,1,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
4,2010,1,1,1,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [17]:
# Create df of maxcol number_responses per year
year_filled = pd.DataFrame({'year': range(2010, 2021), 'maxcol': range(1, 12)})

# For each year, calculate max row sum and update year_filled
for year in year_filled['year']:
    # Subset rows where 'year__*' matches the current year
    subset = calc[calc['year'] == year]
    
    # Calculate row sums of columns (excluding 'year__*') where values >= 1
    row_sums = (subset.drop(columns='year') >= 1).sum(axis=1)
    
    # Update maxcol with the maximum row sum (use 0 if no rows exist)
    year_filled.loc[year_filled['year'] == year, 'maxcol'] = row_sums.max() if not row_sums.empty else 0

In [18]:
year_filled

,year,maxcol
0,2010,191
1,2011,241
2,2012,249
3,2013,271
4,2014,293
5,2015,300
6,2016,333
7,2017,336
8,2018,379
9,2019,394


In [19]:
(calc.drop(columns='year').tail() >= 1).sum(axis=1)

40999    415
41000    340
41001    339
41002    387
41003    430
dtype: int64

### run this and the calc cells above to add 'maxcol' to df!

In [20]:
# Create a mapping from year to maxcol value
year_to_maxcol = dict(zip(year_filled['year'], year_filled['maxcol']))

# Use map to assign the correct maxcol value based on the year
base2.loc[:,'maxcol'] = base2['year'].map(year_to_maxcol)

df.loc[:,'maxcol'] = df['year'].map(year_to_maxcol)
# nice! thank you perplexity.ai!

base_KPIs.loc[:,'maxcol'] = base_KPIs['year'].map(year_to_maxcol)

/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/3441743117.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,'maxcol'] = base2['year'].map(year_to_maxcol)


In [21]:
# base_KPIs.to_csv('preprocessed_data/base_KPIs_labels_v2_with_maxcol.csv', index=False)

In [22]:
base2.tail()

,account_no,year,respondent,maxcol
40999,34421.0,2020,supply_chain,434
41000,838191.0,2020,supply_chain,434
41001,37896.0,2020,supply_chain,434
41002,71988.0,2020,supply_chain,434
41003,21064.0,2020,supply_chain,434


In [23]:
df.tail()

,year,respondent,name,account_no,column=Discloser ID|sheet=Summary|value=response,column=Country|sheet=Summary Data|value=response,column=Primary industry|sheet=Summary Data|value=response,column=Primary Expansion|sheet=Summary|value=response,column=secondary_expansion|sheet=Summary Data|value=response,column=Complexity|sheet=Summary|value=response,...,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_characters,column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_responses,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|model=climate_specificity|label=spec|value=score_average,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_characters,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_responses,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|model=climate_specificity|label=spec|value=score_average,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_characters,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_responses,maxcol
40999,2020,supply_chain,[ZINWELL CORPORATION],34421.0,None,[China],[Manufacturing],None,None,None,...,0.439978,764.0,34,0.440581,769.0,34,0.438721,782.0,34,434
41000,2020,supply_chain,[ZKL PRINTING],838191.0,None,[China],[Manufacturing],None,None,None,...,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34,434
41001,2020,supply_chain,[Zones],37896.0,None,[United States of America],[Services],None,None,None,...,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34,434
41002,2020,supply_chain,[ZOTEQ],71988.0,None,[China],[Materials],None,None,None,...,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34,434
41003,2020,supply_chain,[Zurich Insurance Group],21064.0,None,[Switzerland],[Services],None,None,None,...,0.438721,782.0,34,0.438721,782.0,34,0.438721,782.0,34,434


In [19]:
# # thank you perplexity
# import time

# start_time = time.strftime("%Y-%m-%d %H:%M:%S")
# print(f"Start time: {start_time}")

# boolean_df = a.apply(
#     lambda col: col.apply(
#         lambda x: str(x).lower().find('yes') > -1 if x is not None else False
#     )
# ) # might just be slower but not break the kernel?

# end_time = time.strftime("%Y-%m-%d %H:%M:%S")
# print(f"End time: {end_time}")

In [ ]:
# should be faster than nested apply, but this breaks kernel!
# boolean_df = a.astype(str).apply(lambda col: col.str.contains('yes', case=False, na=False))

In [7]:
# check boolean_df['column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|model=environmental_claims|value=label']

### trying chunking streaming

In [20]:
# # try faster pattern matching in column names: 
# pattern = 'model=environmental_claims|value=label'
# filtered_cols = df.filter(like=pattern, axis=1).columns # check that it doesn't treat the | as an "OR"


In [21]:
# filtered_cols = filtered_cols[-8:]

In [22]:
# label = 'yes'

In [23]:
# print(f'chunk {label} processed')

In [24]:
# test = df[filtered_cols].iloc[-5:, -8:]
# n = 1  # rows per chunk (885 cols vs. 28000)
# chunks = [test[i:i+n] for i in range(0, len(test), n)]

In [25]:
# filtered_cols

In [26]:
# test

In [ ]:
# start small, test with last 5 rows and 8 columns bc I know there are yes's in there... 
# with chunks of 10 rows each, should take about 6 minutes for env claims

In [27]:
# # chunk df into groups of 10,000 rows, should be about 4 chunks
# import time

# start_time = time.time()
# print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# # try faster pattern matching in column names: 
# pattern = 'model=environmental_claims|value=label'
# filtered_cols = df.filter(like=pattern, axis=1).columns # check that it doesn't treat the | as an "OR"
# filtered_cols = filtered_cols[-8:]
# label = 'yes'

# test = df[filtered_cols].iloc[-5:, -8:]
# n = 1  # rows per chunk (885 cols vs. 28000)
# chunks = [test[i:i+n] for i in range(0, len(test), n)]

# all_chunks = []

# for i, chunk in enumerate(chunks):
    
#     # mask the chunk if contains the label
#     chunk = (
#         chunk[filtered_cols]
#         .astype(str)
#         .replace(r"[\[\]']", '', regex=True)
#         .apply(lambda col: col.str.contains(label, case=False, na=False))
#         .sum(axis=1)
#     )
    
#     print(f"chunk {i} processed")
#     all_chunks.append(chunk)
    
# #     # Output processed chunk to a new file (append mode)
# #     chunk.to_csv(f'preprocessed_data/masked_chunk_{pattern}_{label}.csv', 
# #                  mode='a', 
# #                  header=not pd.io.common.file_exists(f'preprocessed_data/masked_chunk_{pattern}_{label}.csv'), 
# #                  index=False)
# #     print(f'chunk {i} saved')


# # return(all_chunks)
    
# end_time = time.time()
# print(f"End: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# elapsed_seconds = end_time - start_time
# elapsed_hms = time.strftime('%H:%M:%S', time.gmtime(elapsed_seconds))
# print(f'Elapsed: {elapsed_hms}')

# base2_results[f'{pattern}_{label}_sum'] = all_chunks
# base2_results


In [28]:
# len(filtered_cols)

In [ ]:
# chunks = [data[i:i+n] for i in range(0, len(data), n)]

#     for i, chunk in enumerate(chunks):
#         print(i)

# DEFINE FUNCTION to calculate row sums and means of label matches

In [24]:
# thank you perplexity
def process_chunks_by_pattern(
    data, 
    cols,
    pattern, 
    label, 
    n=4000,
):
    """
    Processes chunks of a DataFrame, calculating:
    1. Row sums of label matches
    2. Row means (sums divided by maxcol values)
    
    Args:
        data (pd.DataFrame): Input DataFrame.
        cols (list): Columns to process (pre-filtered by pattern).
        pattern (str): [Optional] Pattern used to filter cols.
        label (str): Text label to search for.
        n (int): Rows per chunk.
    
    Returns:
        tuple: (list_of_sum_series, list_of_mean_series)
    """
       
    row_sums = []
    row_means = []
    chunks = [data[i:i+n] for i in range(0, len(data), n)]

    for i, chunk in enumerate(chunks):
        
        # Mask and count label occurrences
        chunk_sum = (
            chunk[cols]
            .astype(str)
            .replace(r"[\[\]']", '', regex=True)
            .apply(lambda col: col.str.contains(label, case=False, na=False))
            .sum(axis=1)
        )
                
        # Calculate means (aligned by index)
        divisor = chunk['maxcol'].replace(0, 1)  # Avoid division by zero
        chunk_mean = chunk_sum / divisor
        
        print(f"chunk {i} processed")
        row_sums.append(chunk_sum)
        row_means.append(chunk_mean)
    
    ## Concatenate ONCE after all chunks processed
    row_sums_concat = pd.concat(row_sums, ignore_index=True)
    row_means_concat = pd.concat(row_means, ignore_index=True)

    return row_sums_concat, row_means_concat

In [30]:
# base2_results[f'{pattern}_{label}_sum'] = process_chunks_by_pattern(test,filtered_cols, pattern, label, n=410)

In [31]:
# base2_results

In [32]:
# df[df.filter(like='number_characters', axis=1).columns].map(lambda x: isinstance(x, (int, float)) and x >= 1).sum(axis=1) 

0        114
1        127
2        166
3        157
4        111
        ... 
40999    415
41000    340
41001    339
41002    387
41003    430
Length: 41004, dtype: int64

In [33]:
# df[df.filter(like='number_characters', axis=1).columns].tail() # issue to target in merge? Question not applicable many times?

,column=Organization|sheet=Summary Data|value=number_characters,column=Account number|sheet=*|value=number_characters,column=Discloser ID|sheet=Summary|value=number_characters,column=Country|sheet=Summary Data|value=number_characters,column=Primary industry|sheet=Summary Data|value=number_characters,column=Primary Expansion|sheet=Summary|value=number_characters,column=secondary_expansion|sheet=Summary Data|value=number_characters,column=Complexity|sheet=Summary|value=number_characters,column=response_status|sheet=Summary Data|value=number_characters,column=Access|sheet=Summary Data|value=number_characters,...,"column=C4.2b_C6_Provide details of any other climate-related targets, including methane reduction targets. - Target denominator (intensity targets only)|sheet=C4.2b|value=number_characters","column=C4.2b_C11_Provide details of any other climate-related targets, including methane reduction targets. - Figure or percentage in reporting year|sheet=C4.2b|value=number_characters","column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|value=number_characters",column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=number_characters,column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|value=number_characters,column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|value=number_characters,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=number_characters",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=number_characters,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=number_characters,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=number_characters
40999,19.0,5.0,NaN,5.0,13.0,NaN,NaN,NaN,NaN,NaN,...,29.0,12.0,6.0,109.0,23.0,15.0,23.0,764.0,769.0,782.0
41000,12.0,6.0,NaN,5.0,13.0,NaN,NaN,NaN,NaN,NaN,...,23.0,0.0,0.0,0.0,23.0,2.0,23.0,782.0,782.0,782.0
41001,5.0,8.0,NaN,24.0,8.0,NaN,NaN,NaN,NaN,NaN,...,23.0,23.0,23.0,30.0,23.0,12.0,23.0,782.0,782.0,782.0
41002,5.0,4.0,NaN,5.0,9.0,NaN,NaN,NaN,NaN,NaN,...,23.0,23.0,23.0,0.0,23.0,12.0,0.0,782.0,782.0,782.0
41003,22.0,5.0,NaN,11.0,8.0,NaN,NaN,NaN,NaN,NaN,...,64.0,24.0,307.0,94.0,99.0,16.0,23.0,782.0,782.0,782.0


In [45]:
# df['maxcol']

0        191
1        191
2        191
3        191
4        191
        ... 
40999    434
41000    434
41001    434
41002    434
41003    434
Name: maxcol, Length: 41004, dtype: int64

# DEFINE FUNCTION to calculate nchar_sum and prop_filled
that also ignores "Hidden Answer", "Question not applicable" and "None" and "N/A"

In [56]:
# df_smaller = df[[col for col in df.columns if "column="  in col
#   and "sheet=Summary" not in col 
#   and "model=" not in col
#   and "number_characters" not in col
#   and "number_responses" not in col
#  ] + ['maxcol']] # core of the survey.......

# # df[[year_col] + num_filled_cols] 

In [25]:
# thank you perplexity
def calc_nchar_sum_prop_filled(
    data, 
    cols,
    n=4000,
):
    """
    Processes chunks of a DataFrame, calculating:
    1. Number of characters in each cell that are not "Hidden answer, Question not applicable, etc"
    2. Creates a lengths vector for number of characters without spaces, and a nchar_sum vector (axis=1)
    3. Creates a nonzero response map that equals 1 where lengths >= 1, also sums that for sum_filled
    4. Calculates proportion of maxcol that have nonzero responses in a given year (sum_filled / maxcol)
    5. Concatenates the nchar_sum vectors and prop_filled vectors
    
    Args:
        data (pd.DataFrame): Input DataFrame.
        cols (list): Columns to process (pre-filtered by pattern).
        n (int): Rows per chunk.
    
    Returns:
        tuple: (nchar_sum_concat, prop_filled_concat)
    """
       
    nchar_sum = []
    prop_filled = []
    chunks = [data[i:i+n] for i in range(0, len(data), n)]

    for i, chunk in enumerate(chunks):

        # Mask and count label occurrences
        cleaned_chunk = (
            chunk[cols].drop('maxcol', axis=1) 
            .astype(str)
            .replace(r"[\[\]']", '', regex=True)
            .replace(r"Question not applicable", '', regex=True)
            .replace(r"Hidden Answer", '', regex=True)
            .replace(r"None", '', regex=True)
            .replace(r"\n", "", regex=True)
            .replace(r'\s+', '', regex=True)
        )
        #print(cleaned_chunk)

        lengths = cleaned_chunk.map(lambda x: len(x))
        #print(lengths)
        sum_lengths = lengths.sum(axis=1) # total lengths of each cell
        nonzero_mask = lengths >= 1 # check where lengths >= 1
        #print(nonzero_mask)
        nonzero_responses = nonzero_mask.sum(axis=1)

        # Calculate means (aligned by index)
        divisor = chunk['maxcol'].replace(0, 1)  # Avoid division by zero
        filled_proportion = nonzero_responses / divisor

        print(f"chunk {i} processed")
        nchar_sum.append(sum_lengths)
        prop_filled.append(filled_proportion)

    ## Concatenate ONCE after all chunks processed
    nchar_sum_concat = pd.concat(nchar_sum, ignore_index=True)
    prop_filled_concat = pd.concat(prop_filled, ignore_index=True)

    return nchar_sum_concat, prop_filled_concat

In [44]:
# df[[col for col in df.columns if "column="  in col
#   and "sheet=Summary" not in col 
#   and "model=" not in col
#   and "number_characters" not in col
#   and "number_responses" not in col
#  ]]

# # Question not applicable
# # None
# # Hidden Answer

,column=C0.1_Give a general description and introduction to your organization.|sheet=C0 - Introduction|value=response,column=C0.5_Select the option that describes the reporting boundary for which climate-related impacts on your business are being reported. Note that this option should align with your chosen approach for consolidating your GHG inventory.|sheet=C0 - Introduction|value=response,"column=CC0.6 - Modules As part of the request for information on behalf of investors, companies in the electric utility sector, companies in the automobile and auto component manufacturing sector, companies in the oil and gas sector, companies in the information and communications technology sector (ICT) and companies in the food, beverage and tobacco sector (FBT) should complete supplementary questions in addition to the core questionnaire.If you are in these sector groupings, the corresponding sector modules will not appear among the options of question CC0.6 but will automatically appear in the ORS navigation bar when you save this page. If you want to query your classification, please emailrespond@cdp.net.If you have not been presented with a sector module that you consider would be appropriate for your company to answer, please select the module below in CC0.6.|sheet=CC0. Introduction|value=response",column=0.5. Please select if you wish to complete a shorter information request.|sheet=Introduction|value=response,"column=CC0.2 C1 - Reporting YearPlease state the start and end date of the year for which you are reporting data.The current reporting year is the latest/most recent 12-month period for which data is reported. Enter the dates of this year first.We request data for more than one reporting period for some emission accounting questions. Please provide data for the three years prior to the current reporting year if you have not provided this information before, or if this is the first time you have answered CDP information request. (This does not apply if you have been offered and selected the option of answering the shorter questionnaire). If you are going to provide additional years of data, please give the dates of those reporting periods here. Work backwards from the most recent reporting year.Please enter dates in following format: day/month/year (in full i.e. 2001). - Enter Periods that will be disclosed|sheet=CC0.2|value=response",column=CC0.3 C1 - Country list configurationPlease select the countries for which you will be supplying data. - Select country|sheet=CC0.3|value=response,"column=C1.3_C1_Do you provide incentives for the management of climate-related issues, including the attainment of targets? - Provide incentives for the management of climate-related issues|sheet=C1.3|value=response",column=C1.3a_C1_Provide further details on the incentives provided for the management of climate-related issues (do not include the names of individuals). - Entitled to incentive|sheet=C1.3a|value=response,column=C1.3a_C2_Provide further details on the incentives provided for the management of climate-related issues (do not include the names of individuals). - Type of incentive|sheet=C1.3a|value=response,column=CC1.1 - Where is the highest level of direct responsibility for climate change within your organization?|sheet=CC1. Governance|value=response,...,"column=C4.2b_C6_Provide details of any other climate-related targets, including methane reduction targets. - Target denominator (intensity targets only)|sheet=C4.2b|value=response","column=C4.2b_C11_Provide details of any other climate-related targets, including methane reduction targets. - Figure or percentage in reporting year|sheet=C4.2b|value=response","column=C4.2b_C14_Provide details of any other climate-related targets, including methane reduction targets. - Is this target part of an emissions target?|sheet=C4.2b|value=response",column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_

In [57]:
# cols = df_smaller.columns[-8:]
# test = df[cols].iloc[-5:, -8:]
# n = 1  # rows per chunk (885 cols vs. 28000)
# chunks = [test[i:i+n] for i in range(0, len(test), n)]

In [58]:
# test

,column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response,column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|value=response,column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|value=response,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=response",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=response,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=response,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=response,maxcol
40999,"[Other, please specify, Energy efficiency in b...",[Question not applicable],"[Yes, Yes, No, No, No, Yes]",[Question not applicable],"[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...",434
41000,[None],[Question not applicable],"[No, None, None, None, None, None]",[Question not applicable],"[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...",434
41001,[Energy efficiency in buildings],[Question not applicable],"[No, No, No, No, No, No]",[Question not applicable],"[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...",434
41002,[None],[Question not applicable],"[No, No, No, No, No, No]",[None],"[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...",434
41003,"[Low-carbon energy consumption, Company policy...",[Scope 3: Fuel and energy-related activities (...,"[Yes, Yes, Yes, No, No, Yes]",[Question not applicable],"[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...",434


In [79]:
# nchar_sum = []
# prop_filled = []
    
# for i, chunk in enumerate(chunks):

#     # Mask and count label occurrences
#     cleaned_chunk = (
#         chunk[cols].drop('maxcol', axis=1) 
#         .astype(str)
#         .replace(r"[\[\]']", '', regex=True)
#         .replace(r"Question not applicable", '', regex=True)
#         .replace(r"Hidden Answer", '', regex=True)
#         .replace(r"None", '', regex=True)
#         .replace(r"\n", "", regex=True)
#         .replace(r'\s+', '', regex=True)
#     )
#     #print(cleaned_chunk)
    
#     lengths = cleaned_chunk.map(lambda x: len(x))
#     print(lengths)
#     sum_lengths = lengths.sum(axis=1) # total lengths of each cell
#     nonzero_mask = lengths >= 1 # check where lengths >= 1
#     print(nonzero_mask)
#     nonzero_responses = nonzero_mask.sum(axis=1)

#     # Calculate means (aligned by index)
#     divisor = chunk['maxcol'].replace(0, 1)  # Avoid division by zero
#     filled_proportion = nonzero_responses / 7 #divisor

#     print(f"chunk {i} processed")
#     nchar_sum.append(sum_lengths)
#     prop_filled.append(filled_proportion)

# ## Concatenate ONCE after all chunks processed
# nchar_sum_concat = pd.concat(nchar_sum, ignore_index=True)
# prop_filled_concat = pd.concat(prop_filled, ignore_index=True)

# nchar_sum_concat
# prop_filled_concat

       column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response  \
40999                                                 99                                                                                                                               

       column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|value=response  \
40999                                                  0                                                                                                                                                 

       column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|value=response  \
4099

0    0.571429
1    0.142857
2    0.285714
3    0.142857
4    0.428571
dtype: float64

In [80]:
# cleaned_test = (
#         test[cols].drop('maxcol', axis=1) 
#         .astype(str)
#         .replace(r"[\[\]']", '', regex=True)
#         .replace(r"Question not applicable", '', regex=True)
#         .replace(r"Hidden Answer", '', regex=True)
#         .replace(r"None", '', regex=True)
#         .replace(r"\n", "", regex=True)
#         .replace(r'\s+', '', regex=True)
#     )
# cleaned_test.to_csv("preprocessed_data/cleaned_test.csv", index=False, chunksize=100)

In [81]:
# cleaned_test

,column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response,column=C10.1c_C1_Provide further details of the verification/assurance undertaken for your Scope 3 emissions and attach the relevant statements. - Scope 3 category|sheet=C10.1c|value=response,column=C8.2_C1_Select which energy-related activities your organization has undertaken. - Indicate whether your organization undertook this energy-related activity in the reporting year|sheet=C8.2|value=response,"column=C8.2e_C3_Provide details on the electricity, heat, steam, and/or cooling amounts that were accounted for at a zero emission factor in the market-based Scope 2 figure reported in C6.3. - Country/region of consumption of low-carbon electricity, heat, steam or cooling|sheet=C8.2e|value=response",column=C11.1b_C2_Complete the following table for each of the emissions trading schemes you are regulated by. - % of Scope 2 emissions covered by the ETS|sheet=C11.1b|value=response,column=C11.1b_C8_Complete the following table for each of the emissions trading schemes you are regulated by. - Verified Scope 2 emissions in metric tons CO2e|sheet=C11.1b|value=response,column=C11.1c_C3_Complete the following table for each of the tax systems you are regulated by. - % of total Scope 1 emissions covered by tax|sheet=C11.1c|value=response
40999,"Other,pleasespecifyEnergyefficiencyinbuildings...",,YesYesNoNoNoYes,,87.52,9504.45,
41000,,,No,,,,
41001,Energyefficiencyinbuildings,,NoNoNoNoNoNo,,,,
41002,,,NoNoNoNoNoNo,,,,
41003,Low-carbonenergyconsumptionCompanypolicyorbeha...,Scope3:Fuelandenergy-relatedactivities(notincl...,YesYesYesNoNoYes,,,,


### trying patterns

In [26]:
start_time = time.time()
print(f"Start: {time.strftime('%Y-%m-%d %H:%M:%S')}")


base2_results = {}

# new since last run (5/23/25)
df_smaller = df[[col for col in df.columns if "column="  in col
  and "sheet=Summary" not in col 
  and "number_characters" not in col
  and "number_responses" not in col
 ] + ['maxcol']]

df_core_survey = df[[col for col in df.columns if "column="  in col
  and "sheet=Summary" not in col 
  and "model=" not in col
  and "number_characters" not in col
  and "number_responses" not in col
 ] + ['maxcol']]

# first, add nchar_sum and prop_filled columns using function I built above
base2_results['nchar_sum'], base2_results['prop_filled'] = calc_nchar_sum_prop_filled(df_core_survey, df_core_survey.columns)


patterns = [#'label=spec|value=score_average', #'number_characters', 'number_responses',
            'model=environmental_claims|value=label', 'model=climate_commitment|value=label',
            'model=climate_specificity|value=label', 'model=netzero_reduction|value=label', 
            'model=climate_sentiment|value=label', 'model=tcfd|value=label', 
            'model=transition|value=label', 'model=renewable|value=label'
]            

for pattern in patterns:
    print(pattern)
    # Find columns matching the pattern, ignore the |, so set regex = False! or use like=pattern
    
    filtered_cols = df_smaller.filter(like=pattern, axis=1).columns
    print(len(filtered_cols))    # CHECKS: there should be 884/886 columns for each 
    
#     if pattern in ['number_characters', 'number_responses']: # 'label=spec|value=score_average',
        
#         if pattern == 'number_responses':
#             base2_results['num_filled'] = df[filtered_cols].map(lambda x: isinstance(x, (int, float)) and x >= 1).sum(axis=1) 
#             base2_results['prop_filled'] = df[filtered_cols].map(lambda x: isinstance(x, (int, float)) and x >= 1).sum(axis=1)  / df['maxcol']
        
#         if pattern in ['number_characters']:
#             base2_results['nchar_sum'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
#             # I do wonder how this handles the cells that have lists of numbers
        
#         if pattern in ['label=spec|value=score_average']:
#             base2_results['average_specificity_sum'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
#             base2_results['average_specificity_mean'] = df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1) / df['maxcol']

    if pattern in ['model=environmental_claims|value=label', 'model=climate_commitment|value=label']:
        for label in ['yes', 'no']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=climate_specificity|value=label':
        for label in ['spec']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results['percent_specificity'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            # if label == 'spec': / len(filtered_cols) for percent_specificity?
        for label in ['non']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results['percent_non_specificity'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            
    if pattern == 'model=netzero_reduction|value=label':
        for label in ['reduction', 'net-zero']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=climate_sentiment|value=label':
        for label in ['opportunity', 'risk', 'neutral']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'], base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=tcfd|value=label':
        for label in ['strategy', 'risk', 'metrics', 'governance']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=transition|value=label':
        for label in ['LABEL_0', 'LABEL_1', 'LABEL_2']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)

    if pattern == 'model=renewable|value=label':
        for label in ['LABEL_0', 'LABEL_1']:
            print(label)
            base2_results[f'{pattern}_{label}_sum'],base2_results[f'{pattern}_{label}_mean'] = process_chunks_by_pattern(df,filtered_cols, pattern, label, n=4000)
            
base2 = df[['account_no', 'year', 'respondent', 'maxcol']]
base2.loc[:,'year'] = (
    base2['year']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'account_no'] = (
    base2['account_no']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
    .pipe(pd.to_numeric, errors='coerce')     # Convert to numeric
)
base2.loc[:,'respondent'] = (
    base2['respondent']
    .astype(str)
    .str.replace(r"[\[\]']", '', regex=True)  # Remove [ ] and '
)

# Update base2 DataFrame with the new columns
for col_name, values in base2_results.items():
    base2.loc[:,col_name] = values

end_time = time.time()
print(f"End: {time.strftime('%Y-%m-%d %H:%M:%S')}")

elapsed_seconds = end_time - start_time
elapsed_hms = time.strftime('%H:%M:%S', time.gmtime(elapsed_seconds))
print(f'Elapsed: {elapsed_hms}')

Start: 2025-05-28 13:08:29
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
model=environmental_claims|value=label
864
yes
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
no
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
model=climate_commitment|value=label
864
yes
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4 processed
chunk 5 processed
chunk 6 processed
chunk 7 processed
chunk 8 processed
chunk 9 processed
chunk 10 processed
no
chunk 0 processed
chunk 1 processed
chunk 2 processed
chunk 3 processed
chunk 4

/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

End: 2025-05-28 16:56:13
Elapsed: 03:47:43


/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base2.loc[:,col_name] = values
/var/folders/km/qbfhgn7s6y78wdq1sqsfj4fc0000gn/T/ipykernel_57765/2199459185.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc

In [86]:
base2_results

{'average_specificity_sum': 0         51.340492
 1         60.293824
 2         86.432624
 3         66.007453
 4         49.302015
             ...    
 40999    185.411525
 41000    150.573291
 41001    147.456646
 41002    172.636056
 41003    190.903180
 Length: 41004, dtype: float64,
 'average_specificity_mean': 0        0.268798
 1        0.315674
 2        0.452527
 3        0.345589
 4        0.258126
            ...   
 40999    0.427215
 41000    0.346943
 41001    0.339762
 41002    0.397779
 41003    0.439869
 Length: 41004, dtype: float64,
 'nchar_sum': 0          5602.0
 1         15817.0
 2         74046.0
 3         60984.0
 4          7277.0
            ...   
 40999     40133.0
 41000     27208.0
 41001     35483.0
 41002     27206.0
 41003    214533.0
 Length: 41004, dtype: float64,
 'num_filled': 0        114
 1        127
 2        166
 3        157
 4        111
         ... 
 40999    415
 41000    340
 41001    339
 41002    387
 41003    430
 Length: 41004, dty

In [87]:
base2.head() # v2 with old prop_filled and nchar_sum

,account_no,year,respondent,average_specificity_sum,average_specificity_mean,nchar_sum,num_filled,prop_filled,model=environmental_claims|value=label_yes_sum,model=environmental_claims|value=label_yes_mean,model=environmental_claims|value=label_no_sum,model=environmental_claims|value=label_no_mean
0,200.0,2010,investor,51.340492,0.268798,5602.0,114,0.596859,2,0.010471,884,4.628272
1,1800.0,2010,investor,60.293824,0.315674,15817.0,127,0.664921,4,0.020942,883,4.623037
2,5300.0,2010,investor,86.432624,0.452527,74046.0,166,0.869110,1,0.005236,886,4.638743
3,29900.0,2010,investor,66.007453,0.345589,60984.0,157,0.821990,7,0.036649,883,4.623037
4,28600.0,2010,investor,49.302015,0.258126,7277.0,111,0.581152,2,0.010471,884,4.628272


In [27]:
base2.head()

,account_no,year,respondent,maxcol,nchar_sum,prop_filled,model=environmental_claims|value=label_yes_sum,model=environmental_claims|value=label_yes_mean,model=environmental_claims|value=label_no_sum,model=environmental_claims|value=label_no_mean,...,model=transition|value=label_LABEL_0_sum,model=transition|value=label_LABEL_0_mean,model=transition|value=label_LABEL_1_sum,model=transition|value=label_LABEL_1_mean,model=transition|value=label_LABEL_2_sum,model=transition|value=label_LABEL_2_mean,model=renewable|value=label_LABEL_0_sum,model=renewable|value=label_LABEL_0_mean,model=renewable|value=label_LABEL_1_sum,model=renewable|value=label_LABEL_1_mean
0,200.0,2010,investor,191,4654,0.549738,2,0.010471,862,4.513089,...,22,0.115183,88,0.460733,2,0.010471,105,0.549738,5,0.026178
1,1800.0,2010,investor,191,13446,0.612565,4,0.020942,861,4.507853,...,31,0.162304,93,0.486911,4,0.020942,115,0.602094,16,0.083770
2,5300.0,2010,investor,191,62986,0.816754,1,0.005236,864,4.523560,...,50,0.261780,108,0.565445,11,0.057592,149,0.780105,20,0.104712
3,29900.0,2010,investor,191,51885,0.774869,7,0.036649,861,4.507853,...,50,0.261780,104,0.544503,9,0.047120,148,0.774869,7,0.036649
4,28600.0,2010,investor,191,6207,0.534031,2,0.010471,862,4.513089,...,28,0.146597,76,0.397906,2,0.010471,98,0.513089,9,0.047120


In [28]:
# remove the model=, and |value=
base2.columns = base2.columns.str.replace('model=', '', regex=False).str.replace('|value=', '_', regex=False)

In [29]:
base2.to_csv('preprocessed_data/base_KPIs_labels_v3.csv', index=False) # v3 has re-calculated nchar_sum and prop_filled

In [101]:
col1 = 'column=CC3.1c C4 - Please also indicate what change in absolute emissions this intensity target reflects - Direction of change anticipated in absolute Scope 3 emissions at target completion?|sheet=CC3.1c|model=environmental_claims|value=label'
col2 = 'column=CC3.1f - Please explain (i) why you do not have a target; and (ii) forecast how your emissions will change over the next five years|sheet=CC3. Targets and Initiatives|model=environmental_claims|value=label'
col3 = 'column=CC3.1b C10 - Please provide details of your intensity target - Comment|sheet=CC3.1b|model=environmental_claims|value=label'

In [102]:
filtered_cols = [col1, col2, col3]
label = 'yes'

In [106]:
df[filtered_cols].astype(str).replace(r"[\[\]']", '', regex=True).apply(lambda col: col.str.contains(label, case=False, na=False)).sum(axis=1)

0        0
1        0
2        0
3        0
4        0
        ..
40999    0
41000    0
41001    0
41002    0
41003    0
Length: 41004, dtype: int64

In [105]:
df[filtered_cols].astype(str).replace(r"[\[\]']", '', regex=True).map(lambda x: label in x.lower()).sum(axis=1)


0        0
1        0
2        0
3        0
4        0
        ..
40999    0
41000    0
41001    0
41002    0
41003    0
Length: 41004, dtype: int64

In [82]:
col2 = 'column=C0.5_Select the option that describes the reporting boundary for which climate-related impacts on your business are being reported. Note that this option should align with your chosen approach for consolidating your GHG inventory.|sheet=C0 - Introduction|value=number_responses'
col3 = 'column=0.5. Please select if you wish to complete a shorter information request.|sheet=Introduction|value=number_responses'
filtered_cols = [col1, col2, col3]
df[filtered_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)

0        1.0
1        1.0
2        1.0
3        1.0
4        1.0
        ... 
40999    1.0
41000    1.0
41001    1.0
41002    1.0
41003    1.0
Length: 41004, dtype: float64

In [31]:
len([col for col in df.columns if "column=C" in col and "sheet=Summary" not in col]) # 2152
len([col for col in df.columns if "column=C" in col and "sheet=Summary" not in col and "model=" not in col]) # 1932
len([col for col in df.columns if "column=C" in col 
     and "sheet=Summary" not in col 
     and "model=" not in col 
     and "number_characters" not in col
     and "number_responses" not in col
    ]) # 644


644

In [36]:
len([col for col in df.columns if "sheet=Summary" in col 
  and "model=" not in col
  and "number_characters" not in col
  and "number_responses" not in col
 ]) # 21


21

In [43]:
len([col for col in df.columns if "column="  in col
  and "sheet=Summary" not in col 
  and "model=" not in col
  and "number_characters" not in col
  and "number_responses" not in col
 ]) # 861 base survey responses

861

In [9]:
[col for col in df.columns if "Account number" in col]

['column=Account number|sheet=*|value=response']

In [5]:
[col for col in df.columns if "column=year|sheet=*|value=meta" in col]

['column=year|sheet=*|value=meta']

In [5]:
[col for col in df.columns if "model=environmental" in col]
# climate_specificity label=spec/non
climate_specificity_cols = [col for col in df.columns if "model=climate_specificity" in col]

# climate_sentiment label=neutral/risk/opportunity
# climate_commitment label=yes/no
# tcfd label=risk/strategy/metrics/governance
# netzero_reduction label=net-zero/reduction
# renewable label=LABEL_1/LABEL_0
# transition label=LABEL_1/LABEL_2/LABEL_0
# environmental_claims label=yes/no

In [9]:
len([col for col in df.columns if "model=" not in col]) #3542 columns, 3654 without model in them

2654

In [16]:
[col for col in df.columns if "model=environmental" in col]
# number_responses
# number_characters

['column=year|sheet=*|model=environmental_claims|value=label',
 'column=year|sheet=*|model=environmental_claims|label=no|value=score',
 'column=year|sheet=*|model=environmental_claims|label=yes|value=score',
 'column=label|sheet=*|model=environmental_claims|value=label',
 'column=label|sheet=*|model=environmental_claims|label=no|value=score',
 'column=label|sheet=*|model=environmental_claims|label=yes|value=score',
 'column=Organization|sheet=Summary Data|model=environmental_claims|value=label',
 'column=Organization|sheet=Summary Data|model=environmental_claims|label=no|value=score',
 'column=Organization|sheet=Summary Data|model=environmental_claims|label=yes|value=score',
 'column=Account number|sheet=*|model=environmental_claims|value=label',
 'column=Account number|sheet=*|model=environmental_claims|label=no|value=score',
 'column=Account number|sheet=*|model=environmental_claims|label=yes|value=score',
 'column=Discloser ID|sheet=Summary|model=environmental_claims|value=label',
 

In [126]:
[col for col in df.columns if "response" in col]#[col for col in df.columns if "Summary Data" in col]
# the stages are:
# Under investigation
# To be implemented*
# Implementation commenced*
# Implemented*
# Not to be implemented

# verification/assurance status
# public/supplier engagement
# emission quantities
# ambition of targets
# internal price on carbon
# internal incentives to "manage climate-related issues"
# frequency withi which climate-related issues are a scheduled agenda item (on the board)

['column=Organization|sheet=Summary Data|value=response',
 'column=Account number|sheet=*|value=response',
 'column=Discloser ID|sheet=Summary|value=response',
 'column=Country|sheet=Summary Data|value=response',
 'column=Primary industry|sheet=Summary Data|value=response',
 'column=Primary Expansion|sheet=Summary|value=response',
 'column=secondary_expansion|sheet=Summary Data|value=response',
 'column=Complexity|sheet=Summary|value=response',
 'column=response_status|sheet=Summary Data|value=response',
 'column=Access|sheet=Summary Data|value=response',
 'column=C0.1_Give a general description and introduction to your organization.|sheet=C0 - Introduction|value=response',
 'column=C0.5_Select the option that describes the reporting boundary for which climate-related impacts on your business are being reported. Note that this option should align with your chosen approach for consolidating your GHG inventory.|sheet=C0 - Introduction|value=response',
 'column=CC0.6 - Modules As part of 

### exporting mini_df of initiative details and payback periods and emissions targets

In [9]:
pd.set_option('display.max_columns', 30) # Shows 20 columns
mini_df = df[['column=Account number|sheet=*|value=response', 'column=year|sheet=*|value=meta',
              "column=label|sheet=*|value=meta",
 'column=Organization|sheet=Summary Data|value=response',
 'column=Primary ISIN|sheet=Summary Data|value=response',
 'column=ISINs|sheet=Summary Data|value=response', 
 'column=C0.4_Select the currency used for all financial information disclosed throughout your response.|sheet=C0 - Introduction|value=response',
 'column=C5.2_Select the name of the standard, protocol, or methodology you have used to collect activity data and calculate emissions.|sheet=C5 - Emissions methodology|value=response',             
# initiatives
 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type|sheet=C4.3b|value=response',
 'column=C4.3b_C1_Provide details on the initiatives implemented in the reporting year in the table below. - Initiative category & Initiative type_G|sheet=C4.3b|value=response',
 'column=C4.3b_C6_Provide details on the initiatives implemented in the reporting year in the table below. - Investment required (unit currency – as specified in C0.4)|sheet=C4.3b|value=response',
 'column=C4.3b_C7_Provide details on the initiatives implemented in the reporting year in the table below. - Payback period|sheet=C4.3b|value=response',
 'column=C4.3b_C8_Provide details on the initiatives implemented in the reporting year in the table below. - Estimated lifetime of the initiative|sheet=C4.3b|value=response',
 'column=C4.3b_C2_Provide details on the initiatives implemented in the reporting year in the table below. - Estimated annual CO2e savings (metric tonnes CO2e)|sheet=C4.3b|value=response',
 'column=C4.3b_C5_Provide details on the initiatives implemented in the reporting year in the table below. - Annual monetary savings (unit currency – as specified in C0.4)|sheet=C4.3b|value=response',
 'column=C4.3b_C4_Provide details on the initiatives implemented in the reporting year in the table below. - Voluntary/Mandatory|sheet=C4.3b|value=response',
 'column=C4.3a_C1_Identify the total number of initiatives at each stage of development, and for those in the implementation stages, the estimated CO2e savings. - Number of initiatives|sheet=C4.3a|value=response',
 'column=C4.3a_C2_Identify the total number of initiatives at each stage of development, and for those in the implementation stages, the estimated CO2e savings. - Total estimated annual CO2e savings in metric tonnes CO2e (only for rows marked *)|sheet=C4.3a|value=response',
 'column=C4.3c_C1_What methods do you use to drive investment in emissions reduction activities? - Method|sheet=C4.3c|value=response',
# emissions
 'column=C4.1b_C14_Provide details of your emissions intensity target(s) and progress made against those target(s). - Intensity figure in reporting year (metric tons CO2e per unit of activity)|sheet=C4.1b|value=response',
 'column=C4.1b_C7_Provide details of your emissions intensity target(s) and progress made against those target(s). - Intensity figure in base year (metric tons CO2e per unit of activity)|sheet=C4.1b|value=response',
 'column=C4.1b_C11_Provide details of your emissions intensity target(s) and progress made against those target(s). - Intensity figure in target year (metric tons CO2e per unit of activity) [auto-calculated]|sheet=C4.1b|value=response',
 'column=C4.1b_C10_Provide details of your emissions intensity target(s) and progress made against those target(s). - Targeted reduction from base year (%)|sheet=C4.1b|value=response',
 'column=C4.1b_C17_Provide details of your emissions intensity target(s) and progress made against those target(s). - Is this a science-based target?|sheet=C4.1b|value=response',
 'column=C4.1b_C6_Provide details of your emissions intensity target(s) and progress made against those target(s). - Base year|sheet=C4.1b|value=response',
 'column=C4.1b_C9_Provide details of your emissions intensity target(s) and progress made against those target(s). - Target year|sheet=C4.1b|value=response',
 'column=C4.1b_C2_Provide details of your emissions intensity target(s) and progress made against those target(s). - Year target was set|sheet=C4.1b|value=response',
              # absolute
 'column=C4.1a_C11_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in reporting year (metric tons CO2e)|sheet=C4.1a|value=response',
 'column=C4.1a_C6_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in base year (metric tons CO2e)|sheet=C4.1a|value=response',
 'column=C4.1a_C10_Provide details of your absolute emissions target(s) and progress made against those targets. - Covered emissions in target year (metric tons CO2e) [auto-calculated]|sheet=C4.1a|value=response',
 'column=C4.1a_C9_Provide details of your absolute emissions target(s) and progress made against those targets. - Targeted reduction from base year (%)|sheet=C4.1a|value=response',
 'column=C4.1a_C14_Provide details of your absolute emissions target(s) and progress made against those targets. - Is this a science-based target?|sheet=C4.1a|value=response',
 'column=C4.1a_C5_Provide details of your absolute emissions target(s) and progress made against those targets. - Base year|sheet=C4.1a|value=response',            
 'column=C4.1a_C8_Provide details of your absolute emissions target(s) and progress made against those targets. - Target year|sheet=C4.1a|value=response',
 'column=C4.1a_C2_Provide details of your absolute emissions target(s) and progress made against those targets. - Year target was set|sheet=C4.1a|value=response'
]]

mini_df.columns = ['account_no', 'year', "respondent", 
                   'organization', 'primary_ISIN', 'ISINs', 'currency', 
                   "emission_calc_protocol",
# initiatives
    "initiative_type", "initiative_type_G", "investment_required", "payback_period",
    "lifetime_of_initiative", "annual_mtCO2e_savings_of_initiative", "annual_monetary_savings_of_initiative",
    "voluntary_or_mandatory", "number_of_initiatives","total_annual_mtCO2e_savings_across_initiatives",
    "methods_to_drive_investment",
# emissions
    "emission_intensity_reporting_year", "emission_intensity_base_year", "emission_intensity_target_year",
    "percent_targeted_reduction_intensity_from_base_year",
    "science_based_intensity_target","base_year_intensity_target", "target_year_intensity_target", "target_set_year_intensity_target",
                   # absolute
    "emission_absolute_reporting_year", "emission_absolute_base_year", "emission_absolute_target_year",
    "percent_targeted_reduction_absolute_from_base_year",
    "science_based_absolute_target", "base_year_absolute_target", "target_year_absolute_target", "target_set_year_absolute_target"]

# add a stages column to go with the number_of_intiatives and total_annual_

In [10]:
mini_df.tail(10)

,account_no,year,respondent,organization,primary_ISIN,ISINs,currency,emission_calc_protocol,initiative_type,initiative_type_G,investment_required,payback_period,lifetime_of_initiative,annual_mtCO2e_savings_of_initiative,annual_monetary_savings_of_initiative,...,emission_intensity_base_year,emission_intensity_target_year,percent_targeted_reduction_intensity_from_base_year,science_based_intensity_target,base_year_intensity_target,target_year_intensity_target,target_set_year_intensity_target,emission_absolute_reporting_year,emission_absolute_base_year,emission_absolute_target_year,percent_targeted_reduction_absolute_from_base_year,science_based_absolute_target,base_year_absolute_target,target_year_absolute_target,target_set_year_absolute_target
40994,[71868],[2020],[supply_chain],[ZHUHAI SENYANG COLOR PRINTING CO LTD.],[None],[None],[CNY],[ISO 14064-1],[None],[None],[None],[None],[None],[None],[None],...,[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[None],[None],[None],[None],[None],[None],[None],[None]
40995,[31334],[2020],[supply_chain],[Zignago Vetro SpA],[IT0004171440],[IT0004171440],[EUR],[European Union Emission Trading System (EU ET...,"[Process optimization, Lighting, Company fleet...","[Energy efficiency in production processes, En...","[7700000.0, 17500.0, None, 5000000.0, None, 10...","[No payback, <1 year, None, 4-10 years, 4-10 y...","[<1 year, 11-15 years, 3-5 years, 6-10 years, ...","[8100.0, 61.0, 89.0, 1385.0, 561.0, 35.0]","[200000.0, 23376.0, 0.0, 431000.0, 210.0, 1300...",...,"[0.647, 0.647]","[0.64053, 0.5329986]","[1.0, 17.62]","[No, but we anticipate setting one in the next...","[2019, 2019]","[2020, 2025]","[2019, 2019]","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica...","[Question not applicable, Question not applica..."
40996,[47483],[2020],[supply_chain],[ZIM Shipping],[None],[None],[USD],[The Greenhouse Gas Protocol: A Corporate Acco...,"[Process optimization, Solid biofuels, Other, ...","[Energy efficiency in production processes, Lo...","[170000.0, 2000000.0, 50000000.0]","[1-3 years, 4-10 years, 4-10 years]","[3-5 years, 3-5 years, 6-10 years]","[440300.0, 230000.0, 519980.0]","[350000.0, 0.0, 0.0]",...,[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[3047696.0],[3988516.0],[1994258.0],[50.0],"[Yes, this target has been approved as science...",[2016],[2030],[2016]
40997,[828452],[2020],[supply_chain],"[ZIMAG TECHNOLOGY CO., LTD.]",[None],[None],[USD],[Taiwan - GHG Reduction Act],"[Waste reduction, Motors and drives, Reuse of ...","[Waste reduction and material circularity, Ene...","[0.0, 10000.0, 30000.0]","[<1 year, 1-3 years, >25 years]","[6-10 years, 16-20 years, >30 years]","[1.8, 400.0, 0.27]","[1000.0, 7500.0, 480.0]",...,[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],[Question not applicable],"[1531.3, 759.84, 36.94, 82.15, 35.99]","[1790.02, 759.84, 36.94, 82.15, 35.99]","[1700.519, 721.848, 35.093, 78.0425, 34.1905]","[5.0, 5.0, 5.0, 5.0, 5.0]","[Yes, we consider this a science-based target,...","[2018, 2019, 2019, 2019, 2019]","[2025, 2030, 2030, 2030, 2030]","[2019, 2020, 2020, 2020, 2020]"
40998,[70105],[2020],[supply_chain],[ZINCOL OSSIDI],[None],[None],[EUR],"[Other, please specify: DECRETO LEGISLATIVO 3 ...",[Process optimization],[Energy efficiency in production processes],[40000.0],[1-3 years],[Ongoing]

In [11]:
mini_df.to_csv('preprocessed_data/mini_df_initiatives_emissions.csv', index=False)


### if all the above doesn't work, try chunking df extraction by model type, and then saving to csv's!
then can append csv's or handle separately in R as before, where the list to str is not such a problem

In [ ]:
# climate_specificity label=spec/non
climate_specificity_cols = [col for col in df.columns if "model=climate_specificity" in col]
climate_sentiment_cols = [col for col in df.columns if "model=climate_sentiment" in col]
climate_commitment_cols = [col for col in df.columns if "model=climate_commitment" in col]
tcfd_cols = [col for col in df.columns if "model=tcfd" in col]
netzero_cols = [col for col in df.columns if "model=netzero" in col]
renewable_cols = [col for col in df.columns if "model=renewable" in col]
transition_cols = [col for col in df.columns if "model=transition" in col]
env_claims_cols = [col for col in df.columns if "model=environmental_claims" in col]

# climate_sentiment label=neutral/risk/opportunity
# climate_commitment label=yes/no
# tcfd label=risk/strategy/metrics/governance
# netzero_reduction label=net-zero/reduction
# renewable label=LABEL_1/LABEL_0
# transition label=LABEL_1/LABEL_2/LABEL_0
# environmental_claims label=yes/no

In [11]:
# df[climate_specificity_cols].to_csv('preprocessed_data/df_BERT_climate_specificity.csv', index=False)
df.to_csv('preprocessed_data/df_BERT_climate_specificity.csv', columns = climate_specificity_cols, index=False)

# 4:40- (for just climate-specificity)
# trying all at once, first then can pull back and do just the 
# this crashed the kernel, maybe because embedded search for columns in df extraction?
# would it also crash if I just tried to extract those 20k columns in vector or list format?
    
# climate_specificity label=spec/non
# climate_sentiment label=neutral/risk/opportunity
# climate_commitment label=yes/no
# tcfd label=risk/strategy/metrics/governance
# netzero_reduction label=net-zero/reduction
# renewable label=LABEL_1/LABEL_0
# transition label=LABEL_1/LABEL_2/LABEL_0
# environmental_claims label=yes/no

KeyboardInterrupt: 

In [ ]:
df_BERT.to_csv('preprocessed_data/BERT_metrics.csv', index=False)


### exporting mini_df of ISINs for manual matching process 

In [ ]:
mini_df = df[['column=Account number|sheet=*|value=response', 'column=year|sheet=*|value=meta',
   'column=Organization|sheet=Summary Data|value=response',
   'column=Primary ISIN|sheet=Summary Data|value=response',
 'column=ISINs|sheet=Summary Data|value=response']]
mini_df.columns = ['account_no', 'year', 'organization', 'primary_ISIN', 'ISINs']